In [ ]:
import pandas as pd
import numpy as np
import yfinance as yf
import matplotlib.pyplot as plt

# ==============================
# CONFIG
# ==============================

FILE_PATH = "<local-path>"

DATE_COL = "Date"
RETURN_COL = "New P/L"
TICKER = "VIXY"

# ==============================
# STEP 1 — LOAD VENDOR DATA
# ==============================

df = pd.read_excel(FILE_PATH)
df[DATE_COL] = pd.to_datetime(df[DATE_COL], errors="coerce")
df = df.dropna(subset=[DATE_COL, RETURN_COL]).copy()
df = df.sort_values(DATE_COL).reset_index(drop=True)
df = df.rename(columns={RETURN_COL: "vendor_return"})

print("Vendor columns:", df.columns.tolist())
print("Vendor rows:", len(df))

# ==============================
# STEP 2 — DOWNLOAD VIXY DATA
# ==============================

start = df[DATE_COL].min().strftime("%Y-%m-%d")
end   = (df[DATE_COL].max() + pd.Timedelta(days=5)).strftime("%Y-%m-%d")

vixy = yf.download(
    TICKER,
    start=start,
    end=end,
    progress=False,
    auto_adjust=False
)

print("\nRaw Yahoo columns:")
print(vixy.columns)

# ==============================
# STEP 3 — FLATTEN MULTIINDEX COLUMNS IF NEEDED
# ==============================

if isinstance(vixy.columns, pd.MultiIndex):
    vixy.columns = [
        col[0] if col[1] == "" else f"{col[0]}_{col[1]}"
        for col in vixy.columns
    ]

print("\nFlattened Yahoo columns:")
print(vixy.columns.tolist())

vixy = vixy.reset_index()

# Sometimes Date can also come as a tuple-like column name after reset_index
if "Date" not in vixy.columns:
    possible_date_cols = [c for c in vixy.columns if "Date" in str(c)]
    if len(possible_date_cols) == 0:
        raise Exception("Could not find Date column in Yahoo data.")
    vixy = vixy.rename(columns={possible_date_cols[0]: "Date"})

# ==============================
# STEP 4 — FIND PRICE COLUMN
# ==============================

possible_price_cols = [
    "Adj Close",
    "Close",
    f"Adj Close_{TICKER}",
    f"Close_{TICKER}",
]

price_col = None
for c in possible_price_cols:
    if c in vixy.columns:
        price_col = c
        break

if price_col is None:
    raise Exception(f"Could not find price column. Available columns: {vixy.columns.tolist()}")

print("\nUsing price column:", price_col)

vixy["Date"] = pd.to_datetime(vixy["Date"], errors="coerce")
vixy = vixy.dropna(subset=["Date", price_col]).copy()
vixy = vixy.sort_values("Date").reset_index(drop=True)

vixy["vixy_return"] = vixy[price_col].pct_change()

vixy = vixy[["Date", "vixy_return"]].dropna().reset_index(drop=True)

print("Yahoo rows after return calc:", len(vixy))

# ==============================
# STEP 5 — MERGE
# ==============================

merged = pd.merge(df, vixy, on="Date", how="inner")
merged = merged.dropna(subset=["vendor_return", "vixy_return"]).reset_index(drop=True)

print("\nMerged rows:", len(merged))

if len(merged) == 0:
    raise Exception("No overlapping dates found between vendor data and VIXY data.")

# ==============================
# STEP 6 — STAT CHECKS
# ==============================

corr = merged["vendor_return"].corr(merged["vixy_return"])
mae = (merged["vendor_return"] - merged["vixy_return"]).abs().mean()
rmse = np.sqrt(((merged["vendor_return"] - merged["vixy_return"]) ** 2).mean())
vol_vendor = merged["vendor_return"].std()
vol_vixy = merged["vixy_return"].std()
vol_ratio = vol_vendor / vol_vixy if vol_vixy != 0 else np.nan

print("\n===== COMPARISON RESULTS =====")
print("Correlation       :", round(corr, 6))
print("MAE               :", round(mae, 6))
print("RMSE              :", round(rmse, 6))
print("Vendor Vol        :", round(vol_vendor, 6))
print("VIXY Vol          :", round(vol_vixy, 6))
print("Vol Ratio         :", round(vol_ratio, 6))

# ==============================
# STEP 7 — CUMULATIVE CURVES
# ==============================

merged["vendor_cum"] = (1 + merged["vendor_return"]).cumprod()
merged["vixy_cum"] = (1 + merged["vixy_return"]).cumprod()

plt.figure(figsize=(12, 6))
plt.plot(merged["Date"], merged["vendor_cum"], label="Vendor Series")
plt.plot(merged["Date"], merged["vixy_cum"], label="Actual VIXY")
plt.title("Vendor Return Series vs VIXY")
plt.xlabel("Date")
plt.ylabel("Cumulative Growth")
plt.legend()
plt.grid(True)
plt.show()

# ==============================
# STEP 8 — SCATTER PLOT
# ==============================

plt.figure(figsize=(7, 7))
plt.scatter(merged["vixy_return"], merged["vendor_return"], alpha=0.5)
plt.xlabel("VIXY Return")
plt.ylabel("Vendor Return")
plt.title(f"Daily Return Scatter | Corr = {corr:.4f}")
plt.grid(True)
plt.show()

# ==============================
# STEP 9 — FINAL INTERPRETATION
# ==============================

print("\n===== INTERPRETATION =====")
if corr > 0.98 and 0.9 <= vol_ratio <= 1.1:
    print("✅ Very strong evidence that the vendor series is VIXY (or extremely close to it).")
elif corr > 0.90:
    print("⚠ Strong similarity to VIXY, but may include adjustments, different methodology, or a related volatility product.")
else:
    print("❌ This does not look like VIXY.")

# Optional: save merged comparison
merged.to_excel("<local-path>", index=False)
print("\nSaved comparison file: VIXY_verification_output.xlsx")

In [ ]:
import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import yfinance as yf

from itertools import product
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    roc_auc_score,
    precision_score,
    recall_score
)

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier
from sklearn.ensemble import HistGradientBoostingClassifier

# =========================================================
# CONFIG
# =========================================================

FEATURES_FILE = "<local-path>"
OUTPUT_DIR = "<local-path>"
os.makedirs(OUTPUT_DIR, exist_ok=True)

TICKER = "VIXY"
START_DATE = "2010-01-01"
TEST_START_DATE = "2025-01-01"

DATE_COL_FEATURES = "date"

# ---------------------------------------------------------
# IMPORTANT ALIGNMENT CONFIG
# ---------------------------------------------------------
# We need to send prediction before market close on date T.
# Features for T are available before market open on T.
# Full VIXY daily data for T is NOT known at prediction time.
#
# Therefore:
# - all VIXY autoregressive features must be lagged by 1 extra day
# - target is shifted by 2 days instead of 1
# ---------------------------------------------------------

EXTRA_MARKET_LAG = 1       # do not use same-day close-based VIXY info
TARGET_FORWARD_DAYS = 2    # T+2 in close-based daily indexing

RETURN_THRESHOLD = 0.0

LAG_SETS = [
    [1, 2, 3, 5],
    [1, 2, 3, 5, 10],
    [1, 2, 3, 5, 10, 20],
]

ROLLING_WINDOWS = [
    [3, 5, 10],
    [5, 10, 20],
]

K_BEST_OPTIONS = [20, 40, 60, 100, "all"]
PROB_THRESHOLDS = [0.50, 0.52, 0.55, 0.60]

RANDOM_STATE = 42


# =========================================================
# STEP 1 — LOAD FEATURE DATA
# =========================================================

features = pd.read_csv(FEATURES_FILE)
features[DATE_COL_FEATURES] = pd.to_datetime(features[DATE_COL_FEATURES], errors="coerce")
features = features.dropna(subset=[DATE_COL_FEATURES]).copy()
features = features.sort_values(DATE_COL_FEATURES).reset_index(drop=True)

print("Feature data shape:", features.shape)
print("Feature data date range:", features[DATE_COL_FEATURES].min(), "to", features[DATE_COL_FEATURES].max())


# =========================================================
# STEP 2 — DOWNLOAD VIXY
# =========================================================

vixy = yf.download(
    TICKER,
    start=START_DATE,
    progress=False,
    auto_adjust=False
)

if isinstance(vixy.columns, pd.MultiIndex):
    vixy.columns = [c[0] if c[1] == "" else f"{c[0]}_{c[1]}" for c in vixy.columns]

vixy = vixy.reset_index()

if "Date" not in vixy.columns:
    possible_date_cols = [c for c in vixy.columns if "Date" in str(c)]
    if len(possible_date_cols) == 0:
        raise Exception("Date column not found in VIXY download.")
    vixy = vixy.rename(columns={possible_date_cols[0]: "Date"})

price_col = None
for c in ["Adj Close", "Close", f"Adj Close_{TICKER}", f"Close_{TICKER}"]:
    if c in vixy.columns:
        price_col = c
        break

if price_col is None:
    raise Exception(f"Price column not found. Columns available: {vixy.columns.tolist()}")

vixy["Date"] = pd.to_datetime(vixy["Date"], errors="coerce")
vixy = vixy.dropna(subset=["Date", price_col]).copy()
vixy = vixy.sort_values("Date").reset_index(drop=True)

vixy["vixy_close"] = vixy[price_col]
vixy["vixy_ret"] = vixy["vixy_close"].pct_change()
vixy["vixy_log_ret"] = np.log(vixy["vixy_close"] / vixy["vixy_close"].shift(1))

print("VIXY data shape:", vixy.shape)
print("VIXY date range:", vixy["Date"].min(), "to", vixy["Date"].max())


# =========================================================
# STEP 3 — MERGE WITH FEATURE DATA
# =========================================================

base = vixy[["Date", "vixy_close", "vixy_ret", "vixy_log_ret"]].copy()

data = pd.merge(
    features,
    base,
    left_on=DATE_COL_FEATURES,
    right_on="Date",
    how="inner"
)

# Keep a single Date column
data["Date"] = pd.to_datetime(data["Date"], errors="coerce")
if DATE_COL_FEATURES in data.columns:
    data = data.drop(columns=[DATE_COL_FEATURES])

data = data.sort_values("Date").reset_index(drop=True)

print("Merged data shape:", data.shape)
print("Merged date range:", data["Date"].min(), "to", data["Date"].max())


# =========================================================
# STEP 4 — BUILD DECISION-TIME-SAFE FEATURES
# =========================================================

def add_vixy_features_decision_safe(df, lag_set, rolling_windows,
                                    extra_market_lag=1,
                                    target_forward_days=2):
    out = df.copy()

    # Base series are close-based. Since same-day close is not known at decision time,
    # every VIXY-derived feature gets one extra lag.
    for lag in lag_set:
        safe_lag = lag + extra_market_lag

        out[f"ret_lag_{lag}"] = out["vixy_ret"].shift(safe_lag)
        out[f"logret_lag_{lag}"] = out["vixy_log_ret"].shift(safe_lag)
        out[f"close_lag_{lag}"] = out["vixy_close"].shift(safe_lag)

    for w in rolling_windows:
        # rolling features computed first, then shifted to ensure only known info is used
        out[f"ret_mean_{w}"] = out["vixy_ret"].rolling(w).mean().shift(extra_market_lag)
        out[f"ret_std_{w}"] = out["vixy_ret"].rolling(w).std().shift(extra_market_lag)
        out[f"price_ma_{w}"] = out["vixy_close"].rolling(w).mean().shift(extra_market_lag)
        out[f"price_vs_ma_{w}"] = (
            out["vixy_close"] / out["vixy_close"].rolling(w).mean() - 1
        ).shift(extra_market_lag)
        out[f"cumret_{w}"] = (
            (1 + out["vixy_ret"]).rolling(w).apply(np.prod, raw=True) - 1
        ).shift(extra_market_lag)

    # Forward target
    out["target_ret_fwd"] = out["vixy_ret"].shift(-target_forward_days)
    out["target_up"] = (out["target_ret_fwd"] > RETURN_THRESHOLD).astype(int)

    return out


# =========================================================
# STEP 5 — MODELS
# =========================================================

def get_models():
    models = {}

    models["logreg_l2"] = LogisticRegression(
        C=1.0,
        penalty="l2",
        solver="liblinear",
        random_state=RANDOM_STATE,
        max_iter=500
    )

    models["logreg_l1"] = LogisticRegression(
        C=0.5,
        penalty="l1",
        solver="liblinear",
        random_state=RANDOM_STATE,
        max_iter=500
    )

    models["rf_300"] = RandomForestClassifier(
        n_estimators=300,
        max_depth=6,
        min_samples_leaf=10,
        random_state=RANDOM_STATE,
        n_jobs=-1
    )

    models["rf_600"] = RandomForestClassifier(
        n_estimators=600,
        max_depth=8,
        min_samples_leaf=5,
        random_state=RANDOM_STATE,
        n_jobs=-1
    )

    models["et_300"] = ExtraTreesClassifier(
        n_estimators=300,
        max_depth=6,
        min_samples_leaf=10,
        random_state=RANDOM_STATE,
        n_jobs=-1
    )

    models["et_600"] = ExtraTreesClassifier(
        n_estimators=600,
        max_depth=8,
        min_samples_leaf=5,
        random_state=RANDOM_STATE,
        n_jobs=-1
    )

    models["gb"] = GradientBoostingClassifier(
        n_estimators=200,
        learning_rate=0.03,
        max_depth=3,
        random_state=RANDOM_STATE
    )

    models["hgb"] = HistGradientBoostingClassifier(
        learning_rate=0.03,
        max_depth=4,
        max_iter=300,
        random_state=RANDOM_STATE
    )

    try:
        from xgboost import XGBClassifier
        models["xgb"] = XGBClassifier(
            n_estimators=400,
            max_depth=4,
            learning_rate=0.03,
            subsample=0.9,
            colsample_bytree=0.8,
            eval_metric="logloss",
            random_state=RANDOM_STATE,
            n_jobs=-1
        )
    except Exception:
        pass

    try:
        from lightgbm import LGBMClassifier
        models["lgbm"] = LGBMClassifier(
            n_estimators=400,
            learning_rate=0.03,
            max_depth=4,
            random_state=RANDOM_STATE,
            n_jobs=-1
        )
    except Exception:
        pass

    return models


# =========================================================
# STEP 6 — PIPELINE AND METRICS
# =========================================================

def evaluate_predictions(y_true, y_prob, threshold=0.50):
    y_pred = (y_prob >= threshold).astype(int)

    out = {
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
    }

    try:
        out["roc_auc"] = roc_auc_score(y_true, y_prob)
    except Exception:
        out["roc_auc"] = np.nan

    return out


def build_pipeline(model, use_scaler, kbest):
    steps = [("imputer", SimpleImputer(strategy="median"))]

    if use_scaler:
        steps.append(("scaler", StandardScaler()))

    if kbest != "all":
        steps.append(("select", SelectKBest(score_func=mutual_info_classif, k=kbest)))

    steps.append(("model", model))
    return Pipeline(steps)


# =========================================================
# STEP 7 — SINGLE EXPERIMENT
# =========================================================

def run_experiment(full_df, lag_set, rolling_windows, feature_mode,
                   model_name, model, use_scaler, kbest):

    df = add_vixy_features_decision_safe(
        full_df.copy(),
        lag_set=lag_set,
        rolling_windows=rolling_windows,
        extra_market_lag=EXTRA_MARKET_LAG,
        target_forward_days=TARGET_FORWARD_DAYS
    )

    # Macro columns = everything except known non-feature fields
    exclude_cols = {
        "Date", "vixy_close", "vixy_ret", "vixy_log_ret",
        "target_ret_fwd", "target_up"
    }

    vixy_engineered_cols = [c for c in df.columns if any(
        c.startswith(prefix) for prefix in [
            "ret_lag_", "logret_lag_", "close_lag_",
            "ret_mean_", "ret_std_", "price_ma_",
            "price_vs_ma_", "cumret_"
        ]
    )]

    macro_cols = [c for c in df.columns if c not in exclude_cols and c not in vixy_engineered_cols]

    if feature_mode == "macro_only":
        feature_cols = macro_cols
    elif feature_mode == "macro_plus_ar":
        feature_cols = macro_cols + [c for c in vixy_engineered_cols if "lag_" in c]
    elif feature_mode == "macro_plus_all_vixy":
        feature_cols = macro_cols + vixy_engineered_cols
    else:
        raise ValueError("Unknown feature_mode")

    use_cols = ["Date", "target_up"] + feature_cols
    df = df[use_cols].copy()

    df = df.dropna(subset=["target_up"]).copy()

    train_df = df[df["Date"] < TEST_START_DATE].copy()
    test_df = df[df["Date"] >= TEST_START_DATE].copy()

    if len(train_df) == 0 or len(test_df) == 0:
        return None, None

    X_train = train_df[feature_cols].copy()
    y_train = train_df["target_up"].astype(int).copy()

    X_test = test_df[feature_cols].copy()
    y_test = test_df["target_up"].astype(int).copy()

    kbest_used = kbest
    if kbest != "all":
        kbest_used = min(kbest, X_train.shape[1])
        if kbest_used <= 0:
            return None, None

    pipe = build_pipeline(model, use_scaler, kbest_used)
    pipe.fit(X_train, y_train)

    if hasattr(pipe, "predict_proba"):
        test_prob = pipe.predict_proba(X_test)[:, 1]
    else:
        z = pipe.decision_function(X_test)
        test_prob = 1 / (1 + np.exp(-z))

    result_rows = []
    pred_detail = test_df[["Date"]].copy()
    pred_detail["y_true"] = y_test.values
    pred_detail["prob_up"] = test_prob

    for thr in PROB_THRESHOLDS:
        metrics = evaluate_predictions(y_test, test_prob, threshold=thr)

        row = {
            "model_name": model_name,
            "feature_mode": feature_mode,
            "use_scaler": use_scaler,
            "kbest": kbest_used,
            "lags": ",".join(map(str, lag_set)),
            "rolls": ",".join(map(str, rolling_windows)),
            "threshold": thr,
            "train_rows": len(train_df),
            "test_rows": len(test_df),
            "n_features_input": X_train.shape[1],
            "class1_train_ratio": y_train.mean(),
            "extra_market_lag": EXTRA_MARKET_LAG,
            "target_forward_days": TARGET_FORWARD_DAYS,
            **metrics
        }
        result_rows.append(row)

    pred_detail["model_name"] = model_name
    pred_detail["feature_mode"] = feature_mode
    pred_detail["use_scaler"] = use_scaler
    pred_detail["kbest"] = kbest_used
    pred_detail["lags"] = ",".join(map(str, lag_set))
    pred_detail["rolls"] = ",".join(map(str, rolling_windows))

    return result_rows, pred_detail


# =========================================================
# STEP 8 — RUN GRID
# =========================================================

models = get_models()
feature_modes = ["macro_only", "macro_plus_ar", "macro_plus_all_vixy"]
scaler_options = [False, True]

all_results = []
all_pred_details = []

experiment_counter = 0

for lag_set, roll_set, feature_mode, (model_name, model), use_scaler, kbest in product(
    LAG_SETS,
    ROLLING_WINDOWS,
    feature_modes,
    models.items(),
    scaler_options,
    K_BEST_OPTIONS
):
    experiment_counter += 1
    print(
        f"Running exp {experiment_counter} | "
        f"{model_name} | {feature_mode} | scaler={use_scaler} | "
        f"kbest={kbest} | lags={lag_set} | rolls={roll_set}"
    )

    result_rows, pred_detail = run_experiment(
        full_df=data,
        lag_set=lag_set,
        rolling_windows=roll_set,
        feature_mode=feature_mode,
        model_name=model_name,
        model=model,
        use_scaler=use_scaler,
        kbest=kbest
    )

    if result_rows is not None:
        all_results.extend(result_rows)
        all_pred_details.append(pred_detail)

    # intermediate save every 20 experiments
    if experiment_counter % 20 == 0 and len(all_results) > 0:
        pd.DataFrame(all_results).to_excel(
            os.path.join(OUTPUT_DIR, "intermediate_results.xlsx"),
            index=False
        )

results_df = pd.DataFrame(all_results)
preds_df = pd.concat(all_pred_details, axis=0, ignore_index=True)

print("\nTotal experiment rows:", len(results_df))


# =========================================================
# STEP 9 — RANK AND SAVE
# =========================================================

results_df = results_df.sort_values(
    ["balanced_accuracy", "accuracy", "roc_auc", "f1"],
    ascending=[False, False, False, False]
).reset_index(drop=True)

print("\nTop 20 experiments:")
print(results_df.head(20))

leaderboard_file = os.path.join(OUTPUT_DIR, "vixy_experiment_leaderboard_tplus2.xlsx")
preds_file = os.path.join(OUTPUT_DIR, "vixy_test_predictions_tplus2.xlsx")

results_df.to_excel(leaderboard_file, index=False)
preds_df.to_excel(preds_file, index=False)

print("\nSaved leaderboard:", leaderboard_file)
print("Saved prediction details:", preds_file)


# =========================================================
# STEP 10 — REFIT BEST MODEL AND GENERATE SIGNALS
# =========================================================

best = results_df.iloc[0].copy()

best_model_name = best["model_name"]
best_feature_mode = best["feature_mode"]
best_use_scaler = bool(best["use_scaler"])
best_kbest = best["kbest"]
best_threshold = float(best["threshold"])
best_lags = list(map(int, str(best["lags"]).split(",")))
best_rolls = list(map(int, str(best["rolls"]).split(",")))

print("\nBest setup selected:")
print(best)

best_model = get_models()[best_model_name]

df_best = add_vixy_features_decision_safe(
    data.copy(),
    lag_set=best_lags,
    rolling_windows=best_rolls,
    extra_market_lag=EXTRA_MARKET_LAG,
    target_forward_days=TARGET_FORWARD_DAYS
)

exclude_cols = {
    "Date", "vixy_close", "vixy_ret", "vixy_log_ret",
    "target_ret_fwd", "target_up"
}

vixy_engineered_cols = [c for c in df_best.columns if any(
    c.startswith(prefix) for prefix in [
        "ret_lag_", "logret_lag_", "close_lag_",
        "ret_mean_", "ret_std_", "price_ma_",
        "price_vs_ma_", "cumret_"
    ]
)]

macro_cols = [c for c in df_best.columns if c not in exclude_cols and c not in vixy_engineered_cols]

if best_feature_mode == "macro_only":
    best_feature_cols = macro_cols
elif best_feature_mode == "macro_plus_ar":
    best_feature_cols = macro_cols + [c for c in vixy_engineered_cols if "lag_" in c]
else:
    best_feature_cols = macro_cols + vixy_engineered_cols

train_full = df_best.dropna(subset=["target_up"]).copy()

X_full = train_full[best_feature_cols].copy()
y_full = train_full["target_up"].astype(int).copy()

best_kbest_used = best_kbest
if best_kbest != "all":
    best_kbest_used = min(int(best_kbest), X_full.shape[1])

best_pipe = build_pipeline(best_model, best_use_scaler, best_kbest_used)
best_pipe.fit(X_full, y_full)

score_df = df_best[["Date", "target_up", "target_ret_fwd"] + best_feature_cols].copy()
score_df = score_df.dropna(subset=best_feature_cols).copy()

X_score = score_df[best_feature_cols].copy()

if hasattr(best_pipe, "predict_proba"):
    score_df["prob_up"] = best_pipe.predict_proba(X_score)[:, 1]
else:
    z = best_pipe.decision_function(X_score)
    score_df["prob_up"] = 1 / (1 + np.exp(-z))

score_df["signal"] = np.where(
    score_df["prob_up"] >= best_threshold, "LONG",
    np.where(score_df["prob_up"] <= (1 - best_threshold), "SHORT", "NO TRADE")
)

latest_signal = score_df.sort_values("Date").tail(20).copy()

latest_signal_file = os.path.join(OUTPUT_DIR, "vixy_latest_signals_tplus2.xlsx")
latest_signal.to_excel(latest_signal_file, index=False)

print("\nSaved latest signals:", latest_signal_file)
print("\nLatest 10 signals:")
print(latest_signal.tail(10)[["Date", "prob_up", "signal"]])

In [ ]:
import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import yfinance as yf

from itertools import product
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    roc_auc_score,
    precision_score,
    recall_score
)

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier

# =========================================================
# CONFIG
# =========================================================

FEATURES_FILE = "<local-path>"
OUTPUT_DIR = "<local-path>"
os.makedirs(OUTPUT_DIR, exist_ok=True)

TICKER = "VIXY"
START_DATE = "2010-01-01"
TEST_START_DATE = "2025-01-01"
DATE_COL_FEATURES = "date"

EXTRA_MARKET_LAG = 1
TARGET_FORWARD_DAYS = 2
RETURN_THRESHOLD = 0.0

# Focused grids based on your leaderboard
LAG_SETS = [
    [1,2,3,5,8,10],
    [1,2,3,5,10,15],
    [1,2,3,5,10,20],
    [1,2,4,6,10,15],
]

ROLLING_WINDOWS = [
    [2,3,5,8,10],
    [3,5,8,10,15],
    [3,5,10,15,20],
    [5,8,10,20],
]

FEATURE_MODES = [
    "macro_plus_all_vixy",
    "macro_plus_ar",
]

K_BEST_OPTIONS = [80, 100, 120, 140, "all"]
PROB_THRESHOLDS = [0.48, 0.50, 0.51, 0.52, 0.53, 0.54, 0.55, 0.56]

RANDOM_STATE = 42


# =========================================================
# LOAD DATA
# =========================================================

features = pd.read_csv(FEATURES_FILE)
features[DATE_COL_FEATURES] = pd.to_datetime(features[DATE_COL_FEATURES], errors="coerce")
features = features.dropna(subset=[DATE_COL_FEATURES]).copy()
features = features.sort_values(DATE_COL_FEATURES).reset_index(drop=True)

vixy = yf.download(
    TICKER,
    start=START_DATE,
    progress=False,
    auto_adjust=False
)

if isinstance(vixy.columns, pd.MultiIndex):
    vixy.columns = [c[0] if c[1] == "" else f"{c[0]}_{c[1]}" for c in vixy.columns]

vixy = vixy.reset_index()

if "Date" not in vixy.columns:
    possible_date_cols = [c for c in vixy.columns if "Date" in str(c)]
    if len(possible_date_cols) == 0:
        raise Exception("Date column not found in VIXY download.")
    vixy = vixy.rename(columns={possible_date_cols[0]: "Date"})

price_col = None
for c in ["Adj Close", "Close", f"Adj Close_{TICKER}", f"Close_{TICKER}"]:
    if c in vixy.columns:
        price_col = c
        break

if price_col is None:
    raise Exception(f"Price column not found. Columns available: {vixy.columns.tolist()}")

vixy["Date"] = pd.to_datetime(vixy["Date"], errors="coerce")
vixy = vixy.dropna(subset=["Date", price_col]).copy()
vixy = vixy.sort_values("Date").reset_index(drop=True)

vixy["vixy_close"] = vixy[price_col]
vixy["vixy_ret"] = vixy["vixy_close"].pct_change()
vixy["vixy_log_ret"] = np.log(vixy["vixy_close"] / vixy["vixy_close"].shift(1))

base = vixy[["Date", "vixy_close", "vixy_ret", "vixy_log_ret"]].copy()

data = pd.merge(
    features,
    base,
    left_on=DATE_COL_FEATURES,
    right_on="Date",
    how="inner"
)

data["Date"] = pd.to_datetime(data["Date"], errors="coerce")
if DATE_COL_FEATURES in data.columns:
    data = data.drop(columns=[DATE_COL_FEATURES])

data = data.sort_values("Date").reset_index(drop=True)

print("Merged data shape:", data.shape)
print("Date range:", data["Date"].min(), "to", data["Date"].max())


# =========================================================
# FEATURE ENGINEERING
# =========================================================

def add_vixy_features_decision_safe(df, lag_set, rolling_windows,
                                    extra_market_lag=1,
                                    target_forward_days=2):
    out = df.copy()

    for lag in lag_set:
        safe_lag = lag + extra_market_lag
        out[f"ret_lag_{lag}"] = out["vixy_ret"].shift(safe_lag)
        out[f"logret_lag_{lag}"] = out["vixy_log_ret"].shift(safe_lag)
        out[f"close_lag_{lag}"] = out["vixy_close"].shift(safe_lag)

    for w in rolling_windows:
        out[f"ret_mean_{w}"] = out["vixy_ret"].rolling(w).mean().shift(extra_market_lag)
        out[f"ret_std_{w}"] = out["vixy_ret"].rolling(w).std().shift(extra_market_lag)
        out[f"price_ma_{w}"] = out["vixy_close"].rolling(w).mean().shift(extra_market_lag)
        out[f"price_vs_ma_{w}"] = (
            out["vixy_close"] / out["vixy_close"].rolling(w).mean() - 1
        ).shift(extra_market_lag)
        out[f"cumret_{w}"] = (
            (1 + out["vixy_ret"]).rolling(w).apply(np.prod, raw=True) - 1
        ).shift(extra_market_lag)

    out["target_ret_fwd"] = out["vixy_ret"].shift(-target_forward_days)
    out["target_up"] = (out["target_ret_fwd"] > RETURN_THRESHOLD).astype(int)

    return out


# =========================================================
# MODELS
# =========================================================

def get_models():
    models = {}

    # Logistic deep tuning
    c_grid = [0.01, 0.03, 0.1, 0.3, 0.5, 1, 2, 3, 5, 10]
    penalties = ["l1", "l2"]
    class_weights = [None, "balanced"]

    for penalty, C, cw in product(penalties, c_grid, class_weights):
        if penalty in ["l1", "l2"]:
            name = f"logreg_{penalty}_C{C}_cw{str(cw)}"
            models[name] = LogisticRegression(
                C=C,
                penalty=penalty,
                solver="liblinear",
                class_weight=cw,
                random_state=RANDOM_STATE,
                max_iter=1000
            )

    # Random Forest focused
    rf_grid = [
        (300, 5, 10),
        (500, 6, 8),
        (700, 8, 5),
        (1000, 10, 3),
    ]
    for n_estimators, max_depth, min_samples_leaf in rf_grid:
        name = f"rf_{n_estimators}_d{max_depth}_leaf{min_samples_leaf}"
        models[name] = RandomForestClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            min_samples_leaf=min_samples_leaf,
            random_state=RANDOM_STATE,
            n_jobs=-1
        )

    # HistGradientBoosting focused
    hgb_grid = [
        (0.01, 3, 200),
        (0.03, 3, 300),
        (0.03, 4, 400),
        (0.05, 4, 300),
        (0.03, 5, 500),
    ]
    for lr, depth, iters in hgb_grid:
        name = f"hgb_lr{lr}_d{depth}_it{iters}"
        models[name] = HistGradientBoostingClassifier(
            learning_rate=lr,
            max_depth=depth,
            max_iter=iters,
            random_state=RANDOM_STATE
        )

    # Optional LightGBM
    try:
        from lightgbm import LGBMClassifier
        lgbm_grid = [
            (300, 3, 0.03),
            (500, 4, 0.03),
            (700, 4, 0.02),
            (500, 5, 0.05),
        ]
        for n_estimators, depth, lr in lgbm_grid:
            name = f"lgbm_{n_estimators}_d{depth}_lr{lr}"
            models[name] = LGBMClassifier(
                n_estimators=n_estimators,
                max_depth=depth,
                learning_rate=lr,
                random_state=RANDOM_STATE,
                n_jobs=-1
            )
    except Exception:
        pass

    return models


# =========================================================
# HELPERS
# =========================================================

def evaluate_predictions(y_true, y_prob, threshold=0.50):
    y_pred = (y_prob >= threshold).astype(int)
    out = {
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
    }
    try:
        out["roc_auc"] = roc_auc_score(y_true, y_prob)
    except Exception:
        out["roc_auc"] = np.nan
    return out


def build_pipeline(model, use_scaler, kbest):
    steps = [("imputer", SimpleImputer(strategy="median"))]

    if use_scaler:
        steps.append(("scaler", StandardScaler()))

    if kbest != "all":
        steps.append(("select", SelectKBest(score_func=mutual_info_classif, k=kbest)))

    steps.append(("model", model))
    return Pipeline(steps)


def run_experiment(full_df, lag_set, rolling_windows, feature_mode,
                   model_name, model, use_scaler, kbest):

    df = add_vixy_features_decision_safe(
        full_df.copy(),
        lag_set=lag_set,
        rolling_windows=rolling_windows,
        extra_market_lag=EXTRA_MARKET_LAG,
        target_forward_days=TARGET_FORWARD_DAYS
    )

    exclude_cols = {
        "Date", "vixy_close", "vixy_ret", "vixy_log_ret",
        "target_ret_fwd", "target_up"
    }

    vixy_engineered_cols = [c for c in df.columns if any(
        c.startswith(prefix) for prefix in [
            "ret_lag_", "logret_lag_", "close_lag_",
            "ret_mean_", "ret_std_", "price_ma_",
            "price_vs_ma_", "cumret_"
        ]
    )]

    macro_cols = [c for c in df.columns if c not in exclude_cols and c not in vixy_engineered_cols]

    if feature_mode == "macro_plus_all_vixy":
        feature_cols = macro_cols + vixy_engineered_cols
    elif feature_mode == "macro_plus_ar":
        feature_cols = macro_cols + [c for c in vixy_engineered_cols if "lag_" in c]
    else:
        raise ValueError("Unknown feature_mode")

    use_cols = ["Date", "target_up"] + feature_cols
    df = df[use_cols].copy()
    df = df.dropna(subset=["target_up"]).copy()

    train_df = df[df["Date"] < TEST_START_DATE].copy()
    test_df = df[df["Date"] >= TEST_START_DATE].copy()

    if len(train_df) == 0 or len(test_df) == 0:
        return None

    X_train = train_df[feature_cols].copy()
    y_train = train_df["target_up"].astype(int).copy()
    X_test = test_df[feature_cols].copy()
    y_test = test_df["target_up"].astype(int).copy()

    kbest_used = kbest
    if kbest != "all":
        kbest_used = min(kbest, X_train.shape[1])
        if kbest_used <= 0:
            return None

    pipe = build_pipeline(model, use_scaler, kbest_used)
    pipe.fit(X_train, y_train)

    if hasattr(pipe, "predict_proba"):
        test_prob = pipe.predict_proba(X_test)[:, 1]
    else:
        z = pipe.decision_function(X_test)
        test_prob = 1 / (1 + np.exp(-z))

    rows = []
    for thr in PROB_THRESHOLDS:
        metrics = evaluate_predictions(y_test, test_prob, threshold=thr)

        rows.append({
            "model_name": model_name,
            "feature_mode": feature_mode,
            "use_scaler": use_scaler,
            "kbest": kbest_used,
            "lags": ",".join(map(str, lag_set)),
            "rolls": ",".join(map(str, rolling_windows)),
            "threshold": thr,
            "train_rows": len(train_df),
            "test_rows": len(test_df),
            "n_features_input": X_train.shape[1],
            "class1_train_ratio": y_train.mean(),
            "extra_market_lag": EXTRA_MARKET_LAG,
            "target_forward_days": TARGET_FORWARD_DAYS,
            **metrics
        })

    return rows


# =========================================================
# RUN ALL
# =========================================================

models = get_models()
all_results = []
experiment_counter = 0

for lag_set, roll_set, feature_mode, (model_name, model), kbest in product(
    LAG_SETS,
    ROLLING_WINDOWS,
    FEATURE_MODES,
    models.items(),
    K_BEST_OPTIONS
):
    use_scaler = True if "logreg" in model_name else False

    experiment_counter += 1
    print(
        f"Running exp {experiment_counter} | {model_name} | "
        f"{feature_mode} | scaler={use_scaler} | kbest={kbest} | "
        f"lags={lag_set} | rolls={roll_set}"
    )

    rows = run_experiment(
        full_df=data,
        lag_set=lag_set,
        rolling_windows=roll_set,
        feature_mode=feature_mode,
        model_name=model_name,
        model=model,
        use_scaler=use_scaler,
        kbest=kbest
    )

    if rows is not None:
        all_results.extend(rows)

    if experiment_counter % 25 == 0 and len(all_results) > 0:
        pd.DataFrame(all_results).to_excel(
            os.path.join(OUTPUT_DIR, "intermediate_results_v2.xlsx"),
            index=False
        )

results_df = pd.DataFrame(all_results)

results_df = results_df.sort_values(
    ["balanced_accuracy", "accuracy", "roc_auc", "f1"],
    ascending=[False, False, False, False]
).reset_index(drop=True)

final_file = os.path.join(OUTPUT_DIR, "vixy_focused_experiments_v2.xlsx")
results_df.to_excel(final_file, index=False)

print("\nSaved:", final_file)
print("\nTop 30:")
print(results_df.head(30).to_string(index=False))

In [ ]:
# =========================================================
# VIXY WALK-FORWARD ROBUSTNESS VALIDATION
# =========================================================
#
# PURPOSE
# -------
# This script validates the selected best VIXY directional model
# using a realistic decision-time-safe framework.
#
# It performs:
#   1. Feature + VIXY data loading
#   2. Decision-time-safe feature engineering
#   3. Walk-forward yearly validation
#   4. Probability bucket testing
#   5. No-trade filter evaluation
#   6. Coefficient stability analysis
#   7. Final refit on all available data
#
# WHY THIS SCRIPT EXISTS
# ----------------------
# We already selected the best model from a grid search.
# Now we need to confirm that:
#   - the model is not leaking information
#   - the performance is stable across years
#   - the model can likely replicate in live conditions
#
# LIVE USE CASE
# -------------
# Features for date T are assumed to be available before market open on T.
# The signal is intended to be sent before market close on T.
# Therefore, same-day close-based VIXY information cannot be used.
#
# To avoid look-ahead bias:
#   - VIXY-derived features are lagged by one extra day
#   - the target is shifted forward by 2 days in daily close-based indexing
#
# =========================================================

import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import yfinance as yf

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score
)
from sklearn.linear_model import LogisticRegression

# =========================================================
# CONFIG
# =========================================================

# ---------- Input / Output ----------
FEATURES_FILE = "<local-path>"  # Path to macro/exogenous features file
OUTPUT_DIR = "<local-path>"


os.makedirs(OUTPUT_DIR, exist_ok=True)

# ---------- Market Data ----------
TICKER = "VIXY"
START_DATE = "2010-01-01"
DATE_COL_FEATURES = "date"

# ---------- Selected Best Model ----------
# Chosen from prior experiment ranking
BEST_FEATURE_MODE = "macro_plus_all_vixy"
BEST_USE_SCALER = True
BEST_KBEST = 120
BEST_LAGS = [1, 2, 4, 6, 10, 15]
BEST_ROLLS = [2, 3, 5, 8, 10]
BEST_THRESHOLD = 0.56

# ---------- Decision-Time Safe Setup ----------
# Features for T known before open of T
# Prediction sent before close of T
# So same-day VIXY close-based return is NOT known
EXTRA_MARKET_LAG = 1
TARGET_FORWARD_DAYS = 2
RETURN_THRESHOLD = 0.0

# ---------- Walk-Forward Validation ----------
# Train on all years before test year
# Test only on the specific year
TEST_YEARS = [2019, 2020, 2021, 2022, 2023, 2024, 2025]

# ---------- General ----------
RANDOM_STATE = 42


# =========================================================
# HELPER FUNCTIONS
# =========================================================

def flatten_yfinance_columns(df: pd.DataFrame) -> pd.DataFrame:
    """
    Flatten yfinance MultiIndex columns if they appear.
    """
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = [c[0] if c[1] == "" else f"{c[0]}_{c[1]}" for c in df.columns]
    return df


def find_price_column(df: pd.DataFrame, ticker: str) -> str:
    """
    Identify the correct price column from yfinance output.
    """
    for c in ["Adj Close", "Close", f"Adj Close_{ticker}", f"Close_{ticker}"]:
        if c in df.columns:
            return c
    raise Exception(f"Price column not found. Available columns: {df.columns.tolist()}")


def add_vixy_features_decision_safe(
    df: pd.DataFrame,
    lag_set: list,
    rolling_windows: list,
    extra_market_lag: int = 1,
    target_forward_days: int = 2
) -> pd.DataFrame:
    """
    Create VIXY-derived lag and rolling features in a decision-time-safe way.

    IMPORTANT
    ---------
    If features for date T are available before open on T and prediction must be
    sent before close on T, then full close-based VIXY return for T is NOT available.
    So every VIXY-based feature must be shifted by one extra day.

    Parameters
    ----------
    df : pd.DataFrame
        Merged dataset containing at least:
        ['Date', 'vixy_close', 'vixy_ret', 'vixy_log_ret']
    lag_set : list
        Lag values to use
    rolling_windows : list
        Rolling windows to use
    extra_market_lag : int
        Extra lag to prevent same-day leakage
    target_forward_days : int
        How many daily return steps ahead to predict

    Returns
    -------
    pd.DataFrame
        Input dataframe with engineered features and target columns added
    """
    out = df.copy()

    # -------------------------
    # Lag features
    # -------------------------
    for lag in lag_set:
        safe_lag = lag + extra_market_lag
        out[f"ret_lag_{lag}"] = out["vixy_ret"].shift(safe_lag)
        out[f"logret_lag_{lag}"] = out["vixy_log_ret"].shift(safe_lag)
        out[f"close_lag_{lag}"] = out["vixy_close"].shift(safe_lag)

    # -------------------------
    # Rolling features
    # -------------------------
    for w in rolling_windows:
        out[f"ret_mean_{w}"] = out["vixy_ret"].rolling(w).mean().shift(extra_market_lag)
        out[f"ret_std_{w}"] = out["vixy_ret"].rolling(w).std().shift(extra_market_lag)
        out[f"price_ma_{w}"] = out["vixy_close"].rolling(w).mean().shift(extra_market_lag)

        out[f"price_vs_ma_{w}"] = (
            out["vixy_close"] / out["vixy_close"].rolling(w).mean() - 1
        ).shift(extra_market_lag)

        out[f"cumret_{w}"] = (
            (1 + out["vixy_ret"]).rolling(w).apply(np.prod, raw=True) - 1
        ).shift(extra_market_lag)

    # -------------------------
    # Forward target
    # -------------------------
    out["target_ret_fwd"] = out["vixy_ret"].shift(-target_forward_days)
    out["target_up"] = (out["target_ret_fwd"] > RETURN_THRESHOLD).astype(int)

    return out


def build_pipeline() -> Pipeline:
    """
    Build the selected best pipeline.

    Selected best model:
    logreg_l2_C0.5_cwNone | macro_plus_all_vixy | scaler=True | kbest=120
    """
    model = LogisticRegression(
        penalty="l2",
        C=0.5,
        class_weight=None,
        solver="liblinear",
        random_state=RANDOM_STATE,
        max_iter=1000
    )

    steps = [
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler() if BEST_USE_SCALER else "passthrough"),
        ("select", SelectKBest(score_func=mutual_info_classif, k=BEST_KBEST)),
        ("model", model)
    ]

    pipe = Pipeline(steps)
    return pipe


def compute_metrics(y_true: pd.Series, y_prob: np.ndarray, threshold: float) -> dict:
    """
    Compute classification metrics using a probability threshold.
    """
    y_pred = (y_prob >= threshold).astype(int)

    out = {
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
    }

    try:
        out["roc_auc"] = roc_auc_score(y_true, y_prob)
    except Exception:
        out["roc_auc"] = np.nan

    return out


# =========================================================
# STEP 1 — LOAD FEATURE DATA
# =========================================================

features = pd.read_csv(FEATURES_FILE)
features[DATE_COL_FEATURES] = pd.to_datetime(features[DATE_COL_FEATURES], errors="coerce")
features = features.dropna(subset=[DATE_COL_FEATURES]).copy()
features = features.sort_values(DATE_COL_FEATURES).reset_index(drop=True)

print("Feature data shape:", features.shape)
print("Feature date range:", features[DATE_COL_FEATURES].min(), "to", features[DATE_COL_FEATURES].max())


# =========================================================
# STEP 2 — DOWNLOAD VIXY DATA
# =========================================================

vixy = yf.download(
    TICKER,
    start=START_DATE,
    progress=False,
    auto_adjust=False
)

vixy = flatten_yfinance_columns(vixy)
vixy = vixy.reset_index()

if "Date" not in vixy.columns:
    possible_date_cols = [c for c in vixy.columns if "Date" in str(c)]
    if len(possible_date_cols) == 0:
        raise Exception("Date column not found in VIXY download.")
    vixy = vixy.rename(columns={possible_date_cols[0]: "Date"})

price_col = find_price_column(vixy, TICKER)

vixy["Date"] = pd.to_datetime(vixy["Date"], errors="coerce")
vixy = vixy.dropna(subset=["Date", price_col]).copy()
vixy = vixy.sort_values("Date").reset_index(drop=True)

vixy["vixy_close"] = vixy[price_col]
vixy["vixy_ret"] = vixy["vixy_close"].pct_change()
vixy["vixy_log_ret"] = np.log(vixy["vixy_close"] / vixy["vixy_close"].shift(1))

print("VIXY data shape:", vixy.shape)
print("VIXY date range:", vixy["Date"].min(), "to", vixy["Date"].max())


# =========================================================
# STEP 3 — MERGE FEATURE DATA + VIXY DATA
# =========================================================

base = vixy[["Date", "vixy_close", "vixy_ret", "vixy_log_ret"]].copy()

data = pd.merge(
    features,
    base,
    left_on=DATE_COL_FEATURES,
    right_on="Date",
    how="inner"
)

data["Date"] = pd.to_datetime(data["Date"], errors="coerce")

if DATE_COL_FEATURES in data.columns:
    data = data.drop(columns=[DATE_COL_FEATURES])

data = data.sort_values("Date").reset_index(drop=True)

print("Merged data shape:", data.shape)
print("Merged date range:", data["Date"].min(), "to", data["Date"].max())


# =========================================================
# STEP 4 — CREATE DECISION-TIME-SAFE MODEL DATASET
# =========================================================

df_model = add_vixy_features_decision_safe(
    data.copy(),
    lag_set=BEST_LAGS,
    rolling_windows=BEST_ROLLS,
    extra_market_lag=EXTRA_MARKET_LAG,
    target_forward_days=TARGET_FORWARD_DAYS
)

exclude_cols = {
    "Date",
    "vixy_close",
    "vixy_ret",
    "vixy_log_ret",
    "target_ret_fwd",
    "target_up"
}

vixy_engineered_cols = [
    c for c in df_model.columns
    if any(
        c.startswith(prefix)
        for prefix in [
            "ret_lag_", "logret_lag_", "close_lag_",
            "ret_mean_", "ret_std_", "price_ma_",
            "price_vs_ma_", "cumret_"
        ]
    )
]

macro_cols = [
    c for c in df_model.columns
    if c not in exclude_cols and c not in vixy_engineered_cols
]

if BEST_FEATURE_MODE == "macro_plus_all_vixy":
    feature_cols = macro_cols + vixy_engineered_cols
elif BEST_FEATURE_MODE == "macro_plus_ar":
    feature_cols = macro_cols + [c for c in vixy_engineered_cols if "lag_" in c]
else:
    raise ValueError("Unsupported feature mode.")

use_cols = ["Date", "target_up", "target_ret_fwd"] + feature_cols
df_model = df_model[use_cols].copy()

# Drop rows where target missing
df_model = df_model.dropna(subset=["target_up"]).reset_index(drop=True)

print("Final modeling dataset shape:", df_model.shape)
print("Total features used before SelectKBest:", len(feature_cols))


# =========================================================
# STEP 5 — WALK-FORWARD VALIDATION
# =========================================================

walkforward_results = []
all_test_predictions = []
coef_stability_rows = []

for test_year in TEST_YEARS:
    print(f"\nRunning walk-forward for test year: {test_year}")

    # -------------------------
    # Split train/test
    # -------------------------
    train_df = df_model[df_model["Date"].dt.year < test_year].copy()
    test_df = df_model[df_model["Date"].dt.year == test_year].copy()

    if len(train_df) == 0 or len(test_df) == 0:
        print(f"Skipping year {test_year}: insufficient data.")
        continue

    X_train = train_df[feature_cols].copy()
    y_train = train_df["target_up"].astype(int).copy()

    X_test = test_df[feature_cols].copy()
    y_test = test_df["target_up"].astype(int).copy()

    # -------------------------
    # Fit model
    # -------------------------
    pipe = build_pipeline()
    pipe.fit(X_train, y_train)

    # -------------------------
    # Predict probabilities
    # -------------------------
    prob_up = pipe.predict_proba(X_test)[:, 1]

    # -------------------------
    # Save yearly metrics
    # -------------------------
    metrics = compute_metrics(y_test, prob_up, BEST_THRESHOLD)
    metrics["test_year"] = test_year
    metrics["train_rows"] = len(train_df)
    metrics["test_rows"] = len(test_df)
    walkforward_results.append(metrics)

    # -------------------------
    # Save test predictions
    # -------------------------
    pred_df = test_df[["Date", "target_up", "target_ret_fwd"]].copy()
    pred_df["test_year"] = test_year
    pred_df["prob_up"] = prob_up
    pred_df["pred_up"] = (pred_df["prob_up"] >= BEST_THRESHOLD).astype(int)

    pred_df["signal"] = np.where(
        pred_df["prob_up"] >= BEST_THRESHOLD,
        "LONG",
        np.where(pred_df["prob_up"] <= (1 - BEST_THRESHOLD), "SHORT", "NO TRADE")
    )

    all_test_predictions.append(pred_df)

    # -------------------------
    # Coefficient stability
    # -------------------------
    # We extract selected features safely because transformed feature counts
    # may differ slightly across train windows.
    selector = pipe.named_steps["select"]
    model = pipe.named_steps["model"]

    feature_cols_used = X_train.columns.tolist()
    selected_mask = selector.get_support()

    if len(selected_mask) != len(feature_cols_used):
        print(
            f"Warning for year {test_year}: selector mask length {len(selected_mask)} "
            f"!= original feature count {len(feature_cols_used)}"
        )
        usable_len = min(len(selected_mask), len(feature_cols_used))
        feature_cols_used = feature_cols_used[:usable_len]
        selected_mask = selected_mask[:usable_len]

    selected_features = np.array(feature_cols_used)[selected_mask]
    model_coefs = model.coef_[0]

    usable_len = min(len(selected_features), len(model_coefs))
    selected_features = selected_features[:usable_len]
    model_coefs = model_coefs[:usable_len]

    coef_df = pd.DataFrame({
        "test_year": test_year,
        "feature": selected_features,
        "coef": model_coefs,
        "abs_coef": np.abs(model_coefs)
    }).sort_values("abs_coef", ascending=False)

    coef_stability_rows.append(coef_df)

    print(f"Finished walk-forward test year: {test_year}")


# =========================================================
# STEP 6 — SAVE WALK-FORWARD OUTPUTS
# =========================================================

walkforward_results_df = pd.DataFrame(walkforward_results)

if len(all_test_predictions) > 0:
    all_test_predictions_df = pd.concat(all_test_predictions, axis=0, ignore_index=True)
else:
    all_test_predictions_df = pd.DataFrame()

if len(coef_stability_rows) > 0:
    coef_stability_df = pd.concat(coef_stability_rows, axis=0, ignore_index=True)
else:
    coef_stability_df = pd.DataFrame()

walkforward_results_df.to_excel(
    os.path.join(OUTPUT_DIR, "walkforward_yearly_results.xlsx"),
    index=False
)

all_test_predictions_df.to_excel(
    os.path.join(OUTPUT_DIR, "walkforward_test_predictions.xlsx"),
    index=False
)

coef_stability_df.to_excel(
    os.path.join(OUTPUT_DIR, "walkforward_coefficient_stability.xlsx"),
    index=False
)

print("\nSaved walk-forward output files.")


# =========================================================
# STEP 7 — WALK-FORWARD SUMMARY
# =========================================================

print("\n========== WALK-FORWARD YEARLY RESULTS ==========")
print(walkforward_results_df)

if len(walkforward_results_df) > 0:
    summary_row = pd.DataFrame([{
        "accuracy_mean": walkforward_results_df["accuracy"].mean(),
        "accuracy_min": walkforward_results_df["accuracy"].min(),
        "accuracy_max": walkforward_results_df["accuracy"].max(),
        "balanced_accuracy_mean": walkforward_results_df["balanced_accuracy"].mean(),
        "balanced_accuracy_min": walkforward_results_df["balanced_accuracy"].min(),
        "balanced_accuracy_max": walkforward_results_df["balanced_accuracy"].max(),
        "f1_mean": walkforward_results_df["f1"].mean(),
        "precision_mean": walkforward_results_df["precision"].mean(),
        "recall_mean": walkforward_results_df["recall"].mean(),
        "roc_auc_mean": walkforward_results_df["roc_auc"].mean(),
    }])

    print("\n========== WALK-FORWARD SUMMARY ==========")
    print(summary_row)


# =========================================================
# STEP 8 — PROBABILITY BUCKET TEST
# =========================================================

if len(all_test_predictions_df) > 0:
    bucket_df = all_test_predictions_df.copy()

    bucket_df["prob_bucket"] = pd.cut(
        bucket_df["prob_up"],
        bins=[0.0, 0.45, 0.50, 0.55, 0.60, 1.00],
        include_lowest=True
    )

    bucket_summary = bucket_df.groupby("prob_bucket").agg(
        n_obs=("target_up", "size"),
        actual_up_rate=("target_up", "mean"),
        avg_prob_up=("prob_up", "mean")
    ).reset_index()

    bucket_summary.to_excel(
        os.path.join(OUTPUT_DIR, "probability_bucket_summary.xlsx"),
        index=False
    )

    print("\n========== PROBABILITY BUCKET SUMMARY ==========")
    print(bucket_summary)


# =========================================================
# STEP 9 — NO-TRADE FILTER TEST
# =========================================================

if len(all_test_predictions_df) > 0:
    signal_df = all_test_predictions_df.copy()

    signal_df["trade_taken"] = signal_df["signal"] != "NO TRADE"

    signal_df["trade_correct"] = np.where(
        signal_df["signal"] == "LONG",
        (signal_df["target_up"] == 1).astype(int),
        np.where(
            signal_df["signal"] == "SHORT",
            (signal_df["target_up"] == 0).astype(int),
            np.nan
        )
    )

    trade_only = signal_df[signal_df["trade_taken"]].copy()

    trade_summary = pd.DataFrame([{
        "total_rows": len(signal_df),
        "trades_taken": len(trade_only),
        "trade_rate": len(trade_only) / len(signal_df) if len(signal_df) > 0 else np.nan,
        "trade_hit_rate": trade_only["trade_correct"].mean() if len(trade_only) > 0 else np.nan,
        "long_trades": (trade_only["signal"] == "LONG").sum(),
        "short_trades": (trade_only["signal"] == "SHORT").sum(),
    }])

    trade_summary.to_excel(
        os.path.join(OUTPUT_DIR, "no_trade_filter_summary.xlsx"),
        index=False
    )

    print("\n========== NO-TRADE FILTER SUMMARY ==========")
    print(trade_summary)


# =========================================================
# STEP 10 — FEATURE STABILITY SUMMARY
# =========================================================

if len(coef_stability_df) > 0:
    top_coef_each_year = (
        coef_stability_df
        .sort_values(["test_year", "abs_coef"], ascending=[True, False])
        .groupby("test_year")
        .head(20)
        .copy()
    )

    feature_stability = top_coef_each_year.groupby("feature").agg(
        years_in_top20=("test_year", "nunique"),
        avg_abs_coef=("abs_coef", "mean")
    ).reset_index().sort_values(
        ["years_in_top20", "avg_abs_coef"],
        ascending=[False, False]
    )

    feature_stability.to_excel(
        os.path.join(OUTPUT_DIR, "feature_stability_summary.xlsx"),
        index=False
    )

    print("\n========== FEATURE STABILITY SUMMARY ==========")
    print(feature_stability.head(30))


# =========================================================
# STEP 11 — FINAL REFIT ON ALL AVAILABLE DATA
# =========================================================

final_train = df_model.copy()

X_final = final_train[feature_cols].copy()
y_final = final_train["target_up"].astype(int).copy()

final_pipe = build_pipeline()
final_pipe.fit(X_final, y_final)

final_score = df_model[["Date", "target_up", "target_ret_fwd"] + feature_cols].copy()
final_score = final_score.dropna(subset=feature_cols).copy()

X_score = final_score[feature_cols].copy()
final_score["prob_up"] = final_pipe.predict_proba(X_score)[:, 1]

final_score["signal"] = np.where(
    final_score["prob_up"] >= BEST_THRESHOLD,
    "LONG",
    np.where(final_score["prob_up"] <= (1 - BEST_THRESHOLD), "SHORT", "NO TRADE")
)

final_score.tail(50).to_excel(
    os.path.join(OUTPUT_DIR, "latest_scored_rows.xlsx"),
    index=False
)

print("\nSaved latest scored rows.")
print("\nLast 10 rows:")
print(final_score.tail(10)[["Date", "prob_up", "signal"]])


# =========================================================
# STEP 12 — SAVE CONFIG SUMMARY
# =========================================================

config_summary = pd.DataFrame([{
    "ticker": TICKER,
    "features_file": FEATURES_FILE,
    "best_feature_mode": BEST_FEATURE_MODE,
    "best_use_scaler": BEST_USE_SCALER,
    "best_kbest": BEST_KBEST,
    "best_lags": ",".join(map(str, BEST_LAGS)),
    "best_rolls": ",".join(map(str, BEST_ROLLS)),
    "best_threshold": BEST_THRESHOLD,
    "extra_market_lag": EXTRA_MARKET_LAG,
    "target_forward_days": TARGET_FORWARD_DAYS,
    "test_years": ",".join(map(str, TEST_YEARS)),
}])

config_summary.to_excel(
    os.path.join(OUTPUT_DIR, "run_config_summary.xlsx"),
    index=False
)

print("\nSaved run configuration summary.")
print("\nProcess complete.")

In [ ]:
# =========================================================
# VIXY + HMM REGIME EXPERIMENT
# =========================================================
#
# PURPOSE
# -------
# This script tests whether Hidden Markov Model (HMM)-based
# volatility regime detection improves the VIXY directional model.
#
# We test 3 variants:
#   1. Base Logistic Model
#   2. Logistic Model + HMM regime features
#   3. Logistic Model + HMM regime filter
#
# WHY HMM
# -------
# VIX/VIXY-type instruments are highly regime dependent.
# Their behavior differs a lot in:
#   - quiet markets
#   - transitional markets
#   - stress / volatility spike markets
#
# HMM can help identify those latent regimes.
#
# LIVE DECISION FRAMEWORK
# -----------------------
# Features for date T are assumed available before market open on T.
# Signal is intended before market close on T.
# Therefore same-day close-based VIXY return cannot be used.
#
# To avoid look-ahead bias:
#   - VIXY-derived features are shifted by one extra day
#   - target is shifted by 2 daily steps
#
# =========================================================

import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import yfinance as yf

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score
)
from sklearn.linear_model import LogisticRegression

# =========================================================
# OPTIONAL HMM IMPORT
# =========================================================
# Install if needed:
# pip install hmmlearn
# =========================================================
try:
    from hmmlearn.hmm import GaussianHMM
except ImportError:
    raise ImportError(
        "hmmlearn is not installed. Install it first using:\n"
        "pip install hmmlearn"
    )

# =========================================================
# CONFIG
# =========================================================

FEATURES_FILE = "<local-path>"
OUTPUT_DIR = "<local-path>"
os.makedirs(OUTPUT_DIR, exist_ok=True)

TICKER = "VIXY"
START_DATE = "2010-01-01"
DATE_COL_FEATURES = "date"

# -------------------------
# Base best model settings
# -------------------------
BEST_FEATURE_MODE = "macro_plus_all_vixy"
BEST_USE_SCALER = True
BEST_KBEST = 120
BEST_LAGS = [1, 2, 4, 6, 10, 15]
BEST_ROLLS = [2, 3, 5, 8, 10]

# -------------------------
# Thresholds
# -------------------------
BASE_THRESHOLD = 0.53
NO_TRADE_UPPER = 0.55
NO_TRADE_LOWER = 0.45

# -------------------------
# Decision-time-safe setup
# -------------------------
EXTRA_MARKET_LAG = 1
TARGET_FORWARD_DAYS = 2
RETURN_THRESHOLD = 0.0

# -------------------------
# Monthly walk-forward
# -------------------------
# We simulate:
# train up to previous month
# test on current month
TEST_START_MONTH = "2020-01-01"

# -------------------------
# HMM setup
# -------------------------
HMM_N_STATES = 3
HMM_COVARIANCE_TYPE = "full"
HMM_N_ITER = 200
HMM_RANDOM_STATE = 42

# HMM input features
# We keep these simple and decision-safe
HMM_WINDOWS = [5, 10, 20]

RANDOM_STATE = 42


# =========================================================
# HELPER FUNCTIONS
# =========================================================

def flatten_yfinance_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Flatten yfinance multi-index columns if needed."""
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = [c[0] if c[1] == "" else f"{c[0]}_{c[1]}" for c in df.columns]
    return df


def find_price_column(df: pd.DataFrame, ticker: str) -> str:
    """Find close/adj close column from yfinance output."""
    for c in ["Adj Close", "Close", f"Adj Close_{ticker}", f"Close_{ticker}"]:
        if c in df.columns:
            return c
    raise Exception(f"Price column not found. Available columns: {df.columns.tolist()}")


def compute_metrics(y_true: pd.Series, y_prob: np.ndarray, threshold: float) -> dict:
    """Compute standard classification metrics using probability threshold."""
    y_pred = (y_prob >= threshold).astype(int)

    out = {
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
    }

    try:
        out["roc_auc"] = roc_auc_score(y_true, y_prob)
    except Exception:
        out["roc_auc"] = np.nan

    return out


def compute_trade_summary(df: pd.DataFrame) -> dict:
    """
    Compute hit rate on traded rows only.
    Requires columns:
      signal, target_up
    """
    out = {}
    temp = df.copy()

    temp["trade_taken"] = temp["signal"] != "NO TRADE"

    temp["trade_correct"] = np.where(
        temp["signal"] == "LONG",
        (temp["target_up"] == 1).astype(int),
        np.where(
            temp["signal"] == "SHORT",
            (temp["target_up"] == 0).astype(int),
            np.nan
        )
    )

    trade_only = temp[temp["trade_taken"]].copy()

    out["total_rows"] = len(temp)
    out["trades_taken"] = len(trade_only)
    out["trade_rate"] = len(trade_only) / len(temp) if len(temp) > 0 else np.nan
    out["trade_hit_rate"] = trade_only["trade_correct"].mean() if len(trade_only) > 0 else np.nan
    out["long_trades"] = (trade_only["signal"] == "LONG").sum()
    out["short_trades"] = (trade_only["signal"] == "SHORT").sum()

    return out


# =========================================================
# DATA LOAD
# =========================================================

features = pd.read_csv(FEATURES_FILE)
features[DATE_COL_FEATURES] = pd.to_datetime(features[DATE_COL_FEATURES], errors="coerce")
features = features.dropna(subset=[DATE_COL_FEATURES]).copy()
features = features.sort_values(DATE_COL_FEATURES).reset_index(drop=True)

print("Feature data shape:", features.shape)
print("Feature date range:", features[DATE_COL_FEATURES].min(), "to", features[DATE_COL_FEATURES].max())

vixy = yf.download(
    TICKER,
    start=START_DATE,
    progress=False,
    auto_adjust=False
)

vixy = flatten_yfinance_columns(vixy)
vixy = vixy.reset_index()

if "Date" not in vixy.columns:
    possible_date_cols = [c for c in vixy.columns if "Date" in str(c)]
    if len(possible_date_cols) == 0:
        raise Exception("Date column not found in VIXY download.")
    vixy = vixy.rename(columns={possible_date_cols[0]: "Date"})

price_col = find_price_column(vixy, TICKER)

vixy["Date"] = pd.to_datetime(vixy["Date"], errors="coerce")
vixy = vixy.dropna(subset=["Date", price_col]).copy()
vixy = vixy.sort_values("Date").reset_index(drop=True)

vixy["vixy_close"] = vixy[price_col]
vixy["vixy_ret"] = vixy["vixy_close"].pct_change()
vixy["vixy_log_ret"] = np.log(vixy["vixy_close"] / vixy["vixy_close"].shift(1))

print("VIXY data shape:", vixy.shape)
print("VIXY date range:", vixy["Date"].min(), "to", vixy["Date"].max())

base = vixy[["Date", "vixy_close", "vixy_ret", "vixy_log_ret"]].copy()

data = pd.merge(
    features,
    base,
    left_on=DATE_COL_FEATURES,
    right_on="Date",
    how="inner"
)

data["Date"] = pd.to_datetime(data["Date"], errors="coerce")
if DATE_COL_FEATURES in data.columns:
    data = data.drop(columns=[DATE_COL_FEATURES])

data = data.sort_values("Date").reset_index(drop=True)

print("Merged data shape:", data.shape)
print("Merged date range:", data["Date"].min(), "to", data["Date"].max())


# =========================================================
# DECISION-TIME-SAFE FEATURE ENGINEERING
# =========================================================

def add_vixy_features_decision_safe(df, lag_set, rolling_windows,
                                    extra_market_lag=1,
                                    target_forward_days=2):
    """
    Build decision-safe VIXY lag and rolling features.
    """
    out = df.copy()

    # Lag features
    for lag in lag_set:
        safe_lag = lag + extra_market_lag
        out[f"ret_lag_{lag}"] = out["vixy_ret"].shift(safe_lag)
        out[f"logret_lag_{lag}"] = out["vixy_log_ret"].shift(safe_lag)
        out[f"close_lag_{lag}"] = out["vixy_close"].shift(safe_lag)

    # Rolling features
    for w in rolling_windows:
        out[f"ret_mean_{w}"] = out["vixy_ret"].rolling(w).mean().shift(extra_market_lag)
        out[f"ret_std_{w}"] = out["vixy_ret"].rolling(w).std().shift(extra_market_lag)
        out[f"price_ma_{w}"] = out["vixy_close"].rolling(w).mean().shift(extra_market_lag)
        out[f"price_vs_ma_{w}"] = (
            out["vixy_close"] / out["vixy_close"].rolling(w).mean() - 1
        ).shift(extra_market_lag)
        out[f"cumret_{w}"] = (
            (1 + out["vixy_ret"]).rolling(w).apply(np.prod, raw=True) - 1
        ).shift(extra_market_lag)

    # Target
    out["target_ret_fwd"] = out["vixy_ret"].shift(-target_forward_days)
    out["target_up"] = (out["target_ret_fwd"] > RETURN_THRESHOLD).astype(int)

    return out


# =========================================================
# HMM FEATURE PREPARATION
# =========================================================

def add_hmm_input_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Create the features used ONLY for HMM regime estimation.
    These must also be decision-time-safe.
    """
    out = df.copy()

    # Base lag-safe return features
    out["hmm_ret_1"] = out["vixy_ret"].shift(1 + EXTRA_MARKET_LAG)
    out["hmm_abs_ret_1"] = out["vixy_ret"].abs().shift(1 + EXTRA_MARKET_LAG)
    out["hmm_logret_1"] = out["vixy_log_ret"].shift(1 + EXTRA_MARKET_LAG)

    for w in HMM_WINDOWS:
        out[f"hmm_ret_mean_{w}"] = out["vixy_ret"].rolling(w).mean().shift(EXTRA_MARKET_LAG)
        out[f"hmm_ret_std_{w}"] = out["vixy_ret"].rolling(w).std().shift(EXTRA_MARKET_LAG)
        out[f"hmm_absret_mean_{w}"] = out["vixy_ret"].abs().rolling(w).mean().shift(EXTRA_MARKET_LAG)

    return out


def fit_hmm_and_add_features(train_df: pd.DataFrame, full_df: pd.DataFrame) -> pd.DataFrame:
    """
    Fit HMM on TRAIN PERIOD ONLY, then score the full dataframe.
    This avoids leakage.

    Returns full_df with:
      hmm_state
      hmm_state_prob
      hmm_prev_state
    """
    out = full_df.copy()

    hmm_feature_cols = [c for c in out.columns if c.startswith("hmm_")]

    # Build train matrix
    train_hmm = train_df[["Date"] + hmm_feature_cols].dropna().copy()
    if len(train_hmm) < 100:
        raise Exception("Not enough data to fit HMM.")

    X_train_hmm = train_hmm[hmm_feature_cols].values

    # Standardize manually for HMM stability
    mu = np.nanmean(X_train_hmm, axis=0)
    sd = np.nanstd(X_train_hmm, axis=0)
    sd[sd == 0] = 1.0

    X_train_scaled = (X_train_hmm - mu) / sd

    hmm = GaussianHMM(
        n_components=HMM_N_STATES,
        covariance_type=HMM_COVARIANCE_TYPE,
        n_iter=HMM_N_ITER,
        random_state=HMM_RANDOM_STATE
    )
    hmm.fit(X_train_scaled)

    # Score full data using train-based scaling
    full_hmm = out[["Date"] + hmm_feature_cols].copy()
    valid_mask = full_hmm[hmm_feature_cols].notna().all(axis=1)

    X_full = full_hmm.loc[valid_mask, hmm_feature_cols].values
    X_full_scaled = (X_full - mu) / sd

    states = hmm.predict(X_full_scaled)
    state_probs = hmm.predict_proba(X_full_scaled)

    # Probability of assigned state
    assigned_state_prob = state_probs[np.arange(len(states)), states]

    out["hmm_state"] = np.nan
    out["hmm_state_prob"] = np.nan

    out.loc[valid_mask, "hmm_state"] = states
    out.loc[valid_mask, "hmm_state_prob"] = assigned_state_prob

    out["hmm_prev_state"] = out["hmm_state"].shift(1)

    return out


# =========================================================
# BASE DATASET
# =========================================================

df_model = add_vixy_features_decision_safe(
    data.copy(),
    lag_set=BEST_LAGS,
    rolling_windows=BEST_ROLLS,
    extra_market_lag=EXTRA_MARKET_LAG,
    target_forward_days=TARGET_FORWARD_DAYS
)

df_model = add_hmm_input_features(df_model)

exclude_cols = {
    "Date", "vixy_close", "vixy_ret", "vixy_log_ret",
    "target_ret_fwd", "target_up"
}

vixy_engineered_cols = [
    c for c in df_model.columns
    if any(
        c.startswith(prefix)
        for prefix in [
            "ret_lag_", "logret_lag_", "close_lag_",
            "ret_mean_", "ret_std_", "price_ma_",
            "price_vs_ma_", "cumret_"
        ]
    )
]

macro_cols = [
    c for c in df_model.columns
    if c not in exclude_cols
    and c not in vixy_engineered_cols
    and not c.startswith("hmm_")
]

base_feature_cols = macro_cols + vixy_engineered_cols
hmm_extra_cols = ["hmm_state", "hmm_state_prob", "hmm_prev_state"]

print("Base feature count:", len(base_feature_cols))


# =========================================================
# PIPELINES
# =========================================================

def build_logistic_pipeline(kbest: int):
    """
    Base logistic pipeline.
    """
    pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("select", SelectKBest(score_func=mutual_info_classif, k=kbest)),
        ("model", LogisticRegression(
            penalty="l2",
            C=0.5,
            class_weight=None,
            solver="liblinear",
            random_state=RANDOM_STATE,
            max_iter=1000
        ))
    ])
    return pipe


# =========================================================
# MONTHLY WALK-FORWARD
# =========================================================

all_months = (
    df_model[df_model["Date"] >= pd.to_datetime(TEST_START_MONTH)]["Date"]
    .dt.to_period("M")
    .drop_duplicates()
    .sort_values()
)

results_rows = []
pred_rows = []

for month_period in all_months:
    test_month_start = month_period.to_timestamp()
    test_month_end = (month_period + 1).to_timestamp()

    train_df = df_model[df_model["Date"] < test_month_start].copy()
    test_df = df_model[
        (df_model["Date"] >= test_month_start) &
        (df_model["Date"] < test_month_end)
    ].copy()

    if len(train_df) < 500 or len(test_df) == 0:
        continue

    # -----------------------------------------------------
    # FIT HMM ON TRAIN ONLY, ADD HMM FEATURES TO TRAIN/TEST
    # -----------------------------------------------------
    train_test_df = pd.concat([train_df, test_df], axis=0).sort_values("Date").reset_index(drop=True)

    try:
        train_test_df = fit_hmm_and_add_features(train_df=train_df, full_df=train_test_df)
    except Exception as e:
        print(f"Skipping month {month_period} due to HMM issue: {e}")
        continue

    # Split back
    train_aug = train_test_df[train_test_df["Date"] < test_month_start].copy()
    test_aug = train_test_df[
        (train_test_df["Date"] >= test_month_start) &
        (train_test_df["Date"] < test_month_end)
    ].copy()

    # =====================================================
    # MODEL A — BASE
    # =====================================================
    use_cols_a = ["Date", "target_up", "target_ret_fwd"] + base_feature_cols
    train_a = train_aug[use_cols_a].dropna(subset=["target_up"]).copy()
    test_a = test_aug[use_cols_a].dropna(subset=["target_up"]).copy()

    X_train_a = train_a[base_feature_cols].copy()
    y_train_a = train_a["target_up"].astype(int).copy()
    X_test_a = test_a[base_feature_cols].copy()
    y_test_a = test_a["target_up"].astype(int).copy()

    pipe_a = build_logistic_pipeline(kbest=min(BEST_KBEST, X_train_a.shape[1]))
    pipe_a.fit(X_train_a, y_train_a)
    prob_a = pipe_a.predict_proba(X_test_a)[:, 1]

    metrics_a = compute_metrics(y_test_a, prob_a, BASE_THRESHOLD)
    metrics_a["month"] = str(month_period)
    metrics_a["model_variant"] = "base_logistic"
    metrics_a["train_rows"] = len(train_a)
    metrics_a["test_rows"] = len(test_a)
    results_rows.append(metrics_a)

    pred_a = test_a[["Date", "target_up", "target_ret_fwd"]].copy()
    pred_a["month"] = str(month_period)
    pred_a["model_variant"] = "base_logistic"
    pred_a["prob_up"] = prob_a
    pred_a["signal"] = np.where(
        pred_a["prob_up"] >= NO_TRADE_UPPER, "LONG",
        np.where(pred_a["prob_up"] <= NO_TRADE_LOWER, "SHORT", "NO TRADE")
    )
    pred_rows.append(pred_a)

    # =====================================================
    # MODEL B — BASE + HMM FEATURES
    # =====================================================
    feature_cols_b = base_feature_cols + hmm_extra_cols

    use_cols_b = ["Date", "target_up", "target_ret_fwd"] + feature_cols_b
    train_b = train_aug[use_cols_b].dropna(subset=["target_up"]).copy()
    test_b = test_aug[use_cols_b].dropna(subset=["target_up"]).copy()

    X_train_b = train_b[feature_cols_b].copy()
    y_train_b = train_b["target_up"].astype(int).copy()
    X_test_b = test_b[feature_cols_b].copy()
    y_test_b = test_b["target_up"].astype(int).copy()

    pipe_b = build_logistic_pipeline(kbest=min(BEST_KBEST, X_train_b.shape[1]))
    pipe_b.fit(X_train_b, y_train_b)
    prob_b = pipe_b.predict_proba(X_test_b)[:, 1]

    metrics_b = compute_metrics(y_test_b, prob_b, BASE_THRESHOLD)
    metrics_b["month"] = str(month_period)
    metrics_b["model_variant"] = "logistic_plus_hmm_features"
    metrics_b["train_rows"] = len(train_b)
    metrics_b["test_rows"] = len(test_b)
    results_rows.append(metrics_b)

    pred_b = test_b[["Date", "target_up", "target_ret_fwd"]].copy()
    pred_b["month"] = str(month_period)
    pred_b["model_variant"] = "logistic_plus_hmm_features"
    pred_b["prob_up"] = prob_b
    pred_b["signal"] = np.where(
        pred_b["prob_up"] >= NO_TRADE_UPPER, "LONG",
        np.where(pred_b["prob_up"] <= NO_TRADE_LOWER, "SHORT", "NO TRADE")
    )
    pred_rows.append(pred_b)

    # =====================================================
    # MODEL C — BASE + HMM REGIME FILTER
    # =====================================================
    # Use base probabilities but only trade in high-confidence HMM states.
    # Here we choose the highest-volatility regime based on train sample.
    # -----------------------------------------------------

    train_hmm_valid = train_aug.dropna(subset=["hmm_state", "hmm_state_prob", "target_up"]).copy()

    # Determine which HMM state is most "volatile" based on avg abs forward return
    regime_rank = (
        train_hmm_valid.groupby("hmm_state")["target_ret_fwd"]
        .apply(lambda x: np.mean(np.abs(x)))
        .sort_values(ascending=False)
    )

    if len(regime_rank) > 0:
        active_regime = regime_rank.index[0]
    else:
        active_regime = None

    pred_c = test_a[["Date", "target_up", "target_ret_fwd"]].copy()
    pred_c["month"] = str(month_period)
    pred_c["model_variant"] = "base_logistic_plus_hmm_filter"
    pred_c["prob_up"] = prob_a

    hmm_test_info = test_aug[["Date", "hmm_state", "hmm_state_prob"]].copy()
    pred_c = pd.merge(pred_c, hmm_test_info, on="Date", how="left")

    pred_c["signal"] = np.where(
        (pred_c["hmm_state"] == active_regime) & (pred_c["prob_up"] >= NO_TRADE_UPPER),
        "LONG",
        np.where(
            (pred_c["hmm_state"] == active_regime) & (pred_c["prob_up"] <= NO_TRADE_LOWER),
            "SHORT",
            "NO TRADE"
        )
    )

    trade_summary_c = compute_trade_summary(pred_c)

    # For filtered model, also create threshold-style binary pred for comparable metrics
    pred_binary_c = np.where(pred_c["prob_up"] >= BASE_THRESHOLD, 1, 0)
    metrics_c = {
        "accuracy": accuracy_score(pred_c["target_up"], pred_binary_c),
        "balanced_accuracy": balanced_accuracy_score(pred_c["target_up"], pred_binary_c),
        "f1": f1_score(pred_c["target_up"], pred_binary_c, zero_division=0),
        "precision": precision_score(pred_c["target_up"], pred_binary_c, zero_division=0),
        "recall": recall_score(pred_c["target_up"], pred_binary_c, zero_division=0),
        "roc_auc": roc_auc_score(pred_c["target_up"], pred_c["prob_up"]) if pred_c["target_up"].nunique() > 1 else np.nan,
        "month": str(month_period),
        "model_variant": "base_logistic_plus_hmm_filter",
        "train_rows": len(train_a),
        "test_rows": len(test_a),
        **trade_summary_c
    }
    results_rows.append(metrics_c)
    pred_rows.append(pred_c)

    print(f"Finished month {month_period}")


# =========================================================
# SAVE RESULTS
# =========================================================

results_df = pd.DataFrame(results_rows)
preds_df = pd.concat(pred_rows, axis=0, ignore_index=True) if len(pred_rows) > 0 else pd.DataFrame()

results_file = os.path.join(OUTPUT_DIR, "monthly_walkforward_hmm_comparison.xlsx")
preds_file = os.path.join(OUTPUT_DIR, "monthly_walkforward_hmm_predictions.xlsx")

results_df.to_excel(results_file, index=False)
preds_df.to_excel(preds_file, index=False)

print("\nSaved:")
print(results_file)
print(preds_file)


# =========================================================
# SUMMARY TABLE
# =========================================================

summary_df = (
    results_df.groupby("model_variant")
    .agg(
        accuracy_mean=("accuracy", "mean"),
        balanced_accuracy_mean=("balanced_accuracy", "mean"),
        f1_mean=("f1", "mean"),
        precision_mean=("precision", "mean"),
        recall_mean=("recall", "mean"),
        roc_auc_mean=("roc_auc", "mean"),
        trades_taken_mean=("trades_taken", "mean"),
        trade_rate_mean=("trade_rate", "mean"),
        trade_hit_rate_mean=("trade_hit_rate", "mean"),
    )
    .reset_index()
)

summary_file = os.path.join(OUTPUT_DIR, "monthly_walkforward_hmm_summary.xlsx")
summary_df.to_excel(summary_file, index=False)

print("\n========== SUMMARY ==========")
print(summary_df)
print("\nSaved summary:", summary_file)

In [ ]:
# =========================================================
# VIXY STABILITY LAB - 5 TO 6 HOUR FOCUSED SEARCH
# =========================================================
#
# PURPOSE
# -------
# This script runs a compact, high-value experiment search for VIXY
# direction prediction with emphasis on:
#
#   1. Stable out-of-sample behavior
#   2. Live-replicable decision-time-safe setup
#   3. Monthly walk-forward retraining
#   4. Practical runtime (~5 to 6 hours, machine dependent)
#
# WHY THIS VERSION
# ----------------
# A previous broad run was too long and likely got interrupted due to:
#   - system restart / update
#   - sleep mode
#   - notebook/kernel shutdown
#
# This version is intentionally smaller and easier to monitor.
#
# LIVE DECISION FRAMEWORK
# -----------------------
# Assumption:
#   - Feature data for date T is available before market open on T
#   - Signal must be produced before market close on T
#
# Therefore:
#   - same-day close-based VIXY data cannot be used
#   - VIXY-derived features are shifted by +1 extra day
#   - target is shifted by +2 daily close-based steps
#
# OUTPUT FILES
# ------------
#   1. full_experiment_summary.xlsx
#   2. full_monthly_detail.xlsx
#   3. full_predictions_detail.xlsx
#   4. ranked_experiments.xlsx
#   5. top5_monthly_detail.xlsx
#   6. skip_log.xlsx
#   7. run_progress_log.xlsx
#
# =========================================================

import os
import time
import warnings
warnings.filterwarnings("ignore")

import traceback
import numpy as np
import pandas as pd
import yfinance as yf

from itertools import product

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.svm import LinearSVC

# Optional libs
HAS_HMM = True
try:
    from hmmlearn.hmm import GaussianHMM
except Exception:
    HAS_HMM = False

HAS_LGBM = True
try:
    from lightgbm import LGBMClassifier
except Exception:
    HAS_LGBM = False


# =========================================================
# CONFIG
# =========================================================

FEATURES_FILE = "<local-path>"
OUTPUT_DIR = "<local-path>"
os.makedirs(OUTPUT_DIR, exist_ok=True)

TICKER = "VIXY"
START_DATE = "2010-01-01"
DATE_COL_FEATURES = "date"

# Decision-time-safe setup
EXTRA_MARKET_LAG = 1
TARGET_FORWARD_DAYS = 2
RETURN_THRESHOLD = 0.0

# Walk-forward
TEST_START_MONTH = "2020-01-01"
MIN_TRAIN_ROWS = 750

# =========================================================
# 5–6 HOUR FOCUSED GRID
# =========================================================

FEATURE_MODES = [
    "macro_plus_ar",
    "macro_plus_all_vixy",
]

LAG_SETS = [
    [1, 2, 4, 6, 10, 15],
]

ROLLING_WINDOWS = [
    [2, 3, 5, 8, 10],
    [3, 5, 8, 10, 15],
]

K_BEST_OPTIONS = [120]

BASE_THRESHOLDS = [0.53]

NO_TRADE_CONFIGS = [
    {"upper": 0.55, "lower": 0.45},
]

REGIME_MODES = ["none", "markov_filter"]
if HAS_HMM:
    REGIME_MODES.append("hmm_filter")

# HMM config
USE_HMM = HAS_HMM
HMM_N_STATES = 3
HMM_N_ITER = 200
HMM_RANDOM_STATE = 42
HMM_WINDOWS = [5, 10, 20]

# Markov proxy
USE_MARKOV_PROXY = True
MARKOV_PROXY_VOL_COL = "mk_vol_10"

RANDOM_STATE = 42
SAVE_EVERY = 3


# =========================================================
# HELPER FUNCTIONS
# =========================================================

def flatten_yfinance_columns(df: pd.DataFrame) -> pd.DataFrame:
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = [c[0] if c[1] == "" else f"{c[0]}_{c[1]}" for c in df.columns]
    return df


def find_price_column(df: pd.DataFrame, ticker: str) -> str:
    for c in ["Adj Close", "Close", f"Adj Close_{ticker}", f"Close_{ticker}"]:
        if c in df.columns:
            return c
    raise Exception(f"Price column not found. Available columns: {df.columns.tolist()}")


def compute_metrics(y_true: pd.Series, y_prob: np.ndarray, threshold: float) -> dict:
    y_pred = (y_prob >= threshold).astype(int)
    out = {
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
    }
    try:
        out["roc_auc"] = roc_auc_score(y_true, y_prob)
    except Exception:
        out["roc_auc"] = np.nan
    return out


def compute_trade_summary(df: pd.DataFrame) -> dict:
    temp = df.copy()
    temp["trade_taken"] = temp["signal"] != "NO TRADE"
    temp["trade_correct"] = np.where(
        temp["signal"] == "LONG",
        (temp["target_up"] == 1).astype(int),
        np.where(
            temp["signal"] == "SHORT",
            (temp["target_up"] == 0).astype(int),
            np.nan
        )
    )
    trade_only = temp[temp["trade_taken"]].copy()
    return {
        "total_rows": len(temp),
        "trades_taken": len(trade_only),
        "trade_rate": len(trade_only) / len(temp) if len(temp) > 0 else np.nan,
        "trade_hit_rate": trade_only["trade_correct"].mean() if len(trade_only) > 0 else np.nan,
        "long_trades": (trade_only["signal"] == "LONG").sum(),
        "short_trades": (trade_only["signal"] == "SHORT").sum(),
    }


def stability_score(summary_row: pd.Series) -> float:
    acc = float(summary_row.get("accuracy_mean", np.nan))
    bacc = float(summary_row.get("balanced_accuracy_mean", np.nan))
    f1 = float(summary_row.get("f1_mean", np.nan))
    roc = float(summary_row.get("roc_auc_mean", np.nan))
    recall = float(summary_row.get("recall_mean", np.nan))
    trade_hit = float(summary_row.get("trade_hit_rate_mean", np.nan))

    acc_std = float(summary_row.get("accuracy_std", np.nan))
    bacc_std = float(summary_row.get("balanced_accuracy_std", np.nan))
    monthly_fail_rate = float(summary_row.get("monthly_fail_rate", np.nan))

    score = 0.0
    if not np.isnan(acc): score += 0.15 * acc
    if not np.isnan(bacc): score += 0.30 * bacc
    if not np.isnan(f1): score += 0.15 * f1
    if not np.isnan(roc): score += 0.20 * roc
    if not np.isnan(recall): score += 0.10 * recall
    if not np.isnan(trade_hit): score += 0.10 * trade_hit

    if not np.isnan(acc_std): score -= 0.10 * acc_std
    if not np.isnan(bacc_std): score -= 0.15 * bacc_std
    if not np.isnan(monthly_fail_rate): score -= 0.20 * monthly_fail_rate

    return score


def add_vixy_features_decision_safe(
    df: pd.DataFrame,
    lag_set: list,
    rolling_windows: list,
    extra_market_lag: int = 1,
    target_forward_days: int = 2
) -> pd.DataFrame:
    out = df.copy()

    for lag in lag_set:
        safe_lag = lag + extra_market_lag
        out[f"ret_lag_{lag}"] = out["vixy_ret"].shift(safe_lag)
        out[f"logret_lag_{lag}"] = out["vixy_log_ret"].shift(safe_lag)
        out[f"close_lag_{lag}"] = out["vixy_close"].shift(safe_lag)

    for w in rolling_windows:
        out[f"ret_mean_{w}"] = out["vixy_ret"].rolling(w).mean().shift(extra_market_lag)
        out[f"ret_std_{w}"] = out["vixy_ret"].rolling(w).std().shift(extra_market_lag)
        out[f"price_ma_{w}"] = out["vixy_close"].rolling(w).mean().shift(extra_market_lag)
        out[f"price_vs_ma_{w}"] = (
            out["vixy_close"] / out["vixy_close"].rolling(w).mean() - 1
        ).shift(extra_market_lag)
        out[f"cumret_{w}"] = (
            (1 + out["vixy_ret"]).rolling(w).apply(np.prod, raw=True) - 1
        ).shift(extra_market_lag)

    out["target_ret_fwd"] = out["vixy_ret"].shift(-target_forward_days)
    out["target_up"] = (out["target_ret_fwd"] > RETURN_THRESHOLD).astype(int)

    return out


def add_hmm_input_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["hmm_ret_1"] = out["vixy_ret"].shift(1 + EXTRA_MARKET_LAG)
    out["hmm_abs_ret_1"] = out["vixy_ret"].abs().shift(1 + EXTRA_MARKET_LAG)
    out["hmm_logret_1"] = out["vixy_log_ret"].shift(1 + EXTRA_MARKET_LAG)

    for w in HMM_WINDOWS:
        out[f"hmm_ret_mean_{w}"] = out["vixy_ret"].rolling(w).mean().shift(EXTRA_MARKET_LAG)
        out[f"hmm_ret_std_{w}"] = out["vixy_ret"].rolling(w).std().shift(EXTRA_MARKET_LAG)
        out[f"hmm_absret_mean_{w}"] = out["vixy_ret"].abs().rolling(w).mean().shift(EXTRA_MARKET_LAG)

    return out


def fit_hmm_and_add_features(train_df: pd.DataFrame, full_df: pd.DataFrame) -> pd.DataFrame:
    out = full_df.copy()

    hmm_feature_cols = [c for c in out.columns if c.startswith("hmm_")]
    train_hmm = train_df[["Date"] + hmm_feature_cols].dropna().copy()

    if len(train_hmm) < 150:
        raise Exception("Not enough rows to fit HMM.")

    X_train = train_hmm[hmm_feature_cols].values
    mu = np.nanmean(X_train, axis=0)
    sd = np.nanstd(X_train, axis=0)
    sd[sd == 0] = 1.0
    X_train_scaled = (X_train - mu) / sd

    hmm = GaussianHMM(
        n_components=HMM_N_STATES,
        covariance_type="full",
        n_iter=HMM_N_ITER,
        random_state=HMM_RANDOM_STATE
    )
    hmm.fit(X_train_scaled)

    valid_mask = out[hmm_feature_cols].notna().all(axis=1)
    X_full = out.loc[valid_mask, hmm_feature_cols].values
    X_full_scaled = (X_full - mu) / sd

    states = hmm.predict(X_full_scaled)
    probs = hmm.predict_proba(X_full_scaled)
    assigned_prob = probs[np.arange(len(states)), states]

    out["hmm_state"] = np.nan
    out["hmm_state_prob"] = np.nan
    out.loc[valid_mask, "hmm_state"] = states
    out.loc[valid_mask, "hmm_state_prob"] = assigned_prob
    out["hmm_prev_state"] = out["hmm_state"].shift(1)

    return out


def add_markov_proxy_inputs(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["mk_ret_safe"] = out["vixy_ret"].shift(1 + EXTRA_MARKET_LAG)
    out["mk_abs_ret_safe"] = out["vixy_ret"].abs().shift(1 + EXTRA_MARKET_LAG)
    out["mk_vol_5"] = out["vixy_ret"].rolling(5).std().shift(EXTRA_MARKET_LAG)
    out["mk_vol_10"] = out["vixy_ret"].rolling(10).std().shift(EXTRA_MARKET_LAG)
    out["mk_vol_20"] = out["vixy_ret"].rolling(20).std().shift(EXTRA_MARKET_LAG)
    return out


def fit_markov_proxy_and_add_features(train_df: pd.DataFrame, full_df: pd.DataFrame) -> pd.DataFrame:
    out = full_df.copy()
    vol_col = MARKOV_PROXY_VOL_COL

    train_valid = train_df.loc[train_df[vol_col].notna(), vol_col].copy()
    if len(train_valid) < 100:
        raise Exception("Not enough rows for markov proxy.")

    q1 = train_valid.quantile(1 / 3)
    q2 = train_valid.quantile(2 / 3)

    valid_mask = out[vol_col].notna()
    vals = out.loc[valid_mask, vol_col]

    bucket_state = np.where(
        vals <= q1, 0,
        np.where(vals <= q2, 1, 2)
    )

    out["markov_state"] = np.nan
    out["markov_state_prob"] = np.nan
    out.loc[valid_mask, "markov_state"] = bucket_state
    out.loc[valid_mask, "markov_state_prob"] = np.where(
        bucket_state == 2, 0.70,
        np.where(bucket_state == 1, 0.55, 0.70)
    )
    out["markov_prev_state"] = out["markov_state"].shift(1)

    return out


def build_model_catalog():
    models = {}

    models["logreg_l2_C0.5"] = {
        "model": LogisticRegression(
            penalty="l2", C=0.5, solver="liblinear",
            max_iter=1000, random_state=RANDOM_STATE
        ),
        "use_scaler": True
    }

    models["logreg_l1_C2"] = {
        "model": LogisticRegression(
            penalty="l1", C=2.0, solver="liblinear",
            max_iter=1000, random_state=RANDOM_STATE
        ),
        "use_scaler": True
    }

    svm = LinearSVC(C=0.5, random_state=RANDOM_STATE, max_iter=5000)
    models["linear_svm_calibrated"] = {
        "model": CalibratedClassifierCV(svm, method="sigmoid", cv=3),
        "use_scaler": True
    }

    models["hgb_lr003_d4"] = {
        "model": HistGradientBoostingClassifier(
            learning_rate=0.03,
            max_depth=4,
            max_iter=300,
            random_state=RANDOM_STATE
        ),
        "use_scaler": False
    }

    if HAS_LGBM:
        models["lgbm_500_d4"] = {
            "model": LGBMClassifier(
                n_estimators=500,
                max_depth=4,
                learning_rate=0.03,
                random_state=RANDOM_STATE,
                n_jobs=-1
            ),
            "use_scaler": False
        }

    return models


def build_pipeline(model, use_scaler: bool, kbest, n_features: int):
    if kbest == "all":
        k_used = "all"
    else:
        k_used = min(int(kbest), n_features)

    steps = [
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler() if use_scaler else "passthrough"),
    ]

    if k_used != "all":
        steps.append(("select", SelectKBest(score_func=mutual_info_classif, k=k_used)))

    steps.append(("model", model))
    return Pipeline(steps), k_used


def seconds_to_hms(seconds: float) -> str:
    if seconds is None or np.isnan(seconds):
        return "N/A"
    seconds = max(0, int(seconds))
    h = seconds // 3600
    m = (seconds % 3600) // 60
    s = seconds % 60
    return f"{h:02d}:{m:02d}:{s:02d}"


# =========================================================
# STEP 1 — LOAD FEATURES
# =========================================================

features = pd.read_csv(FEATURES_FILE)
features[DATE_COL_FEATURES] = pd.to_datetime(features[DATE_COL_FEATURES], errors="coerce")
features = features.dropna(subset=[DATE_COL_FEATURES]).copy()
features = features.sort_values(DATE_COL_FEATURES).reset_index(drop=True)

print("Feature data shape:", features.shape)
print("Feature date range:", features[DATE_COL_FEATURES].min(), "to", features[DATE_COL_FEATURES].max())


# =========================================================
# STEP 2 — DOWNLOAD VIXY
# =========================================================

vixy = yf.download(
    TICKER,
    start=START_DATE,
    progress=False,
    auto_adjust=False
)

vixy = flatten_yfinance_columns(vixy)
vixy = vixy.reset_index()

if "Date" not in vixy.columns:
    possible_date_cols = [c for c in vixy.columns if "Date" in str(c)]
    if len(possible_date_cols) == 0:
        raise Exception("Date column not found in VIXY download.")
    vixy = vixy.rename(columns={possible_date_cols[0]: "Date"})

price_col = find_price_column(vixy, TICKER)

vixy["Date"] = pd.to_datetime(vixy["Date"], errors="coerce")
vixy = vixy.dropna(subset=["Date", price_col]).copy()
vixy = vixy.sort_values("Date").reset_index(drop=True)

vixy["vixy_close"] = vixy[price_col]
vixy["vixy_ret"] = vixy["vixy_close"].pct_change()
vixy["vixy_log_ret"] = np.log(vixy["vixy_close"] / vixy["vixy_close"].shift(1))

print("VIXY data shape:", vixy.shape)
print("VIXY date range:", vixy["Date"].min(), "to", vixy["Date"].max())


# =========================================================
# STEP 3 — MERGE DATA
# =========================================================

base = vixy[["Date", "vixy_close", "vixy_ret", "vixy_log_ret"]].copy()

data = pd.merge(
    features,
    base,
    left_on=DATE_COL_FEATURES,
    right_on="Date",
    how="inner"
)

data["Date"] = pd.to_datetime(data["Date"], errors="coerce")
if DATE_COL_FEATURES in data.columns:
    data = data.drop(columns=[DATE_COL_FEATURES])

data = data.sort_values("Date").reset_index(drop=True)

print("Merged data shape:", data.shape)
print("Merged date range:", data["Date"].min(), "to", data["Date"].max())


# =========================================================
# STEP 4 — BUILD EXPERIMENT LIST
# =========================================================

models_catalog = build_model_catalog()

experiment_grid = list(product(
    FEATURE_MODES,
    LAG_SETS,
    ROLLING_WINDOWS,
    K_BEST_OPTIONS,
    models_catalog.items(),
    BASE_THRESHOLDS,
    NO_TRADE_CONFIGS,
    REGIME_MODES
))

total_experiments = len(experiment_grid)

print("\n========== RUN PLAN ==========")
print(f"Models            : {len(models_catalog)}")
print(f"Feature modes     : {len(FEATURE_MODES)}")
print(f"Lag sets          : {len(LAG_SETS)}")
print(f"Rolling sets      : {len(ROLLING_WINDOWS)}")
print(f"K-best options    : {len(K_BEST_OPTIONS)}")
print(f"Thresholds        : {len(BASE_THRESHOLDS)}")
print(f"No-trade configs  : {len(NO_TRADE_CONFIGS)}")
print(f"Regime modes      : {len(REGIME_MODES)}")
print(f"Total experiments : {total_experiments}")
print("================================\n")


# =========================================================
# STEP 5 — MAIN LOOP
# =========================================================

all_results_rows = []
all_monthly_rows = []
all_pred_rows = []
skip_log = []
progress_rows = []

all_months = (
    data[data["Date"] >= pd.to_datetime(TEST_START_MONTH)]["Date"]
    .dt.to_period("M")
    .drop_duplicates()
    .sort_values()
)

run_start_time = time.time()
finished_exp = 0

for exp_id, (
    feature_mode,
    lag_set,
    roll_set,
    kbest,
    model_tuple,
    base_threshold,
    no_trade_cfg,
    regime_mode
) in enumerate(experiment_grid, start=1):

    exp_start_time = time.time()

    model_name, model_info = model_tuple
    model = model_info["model"]
    use_scaler = model_info["use_scaler"]

    exp_name = (
        f"exp_{exp_id}__{model_name}__{feature_mode}"
        f"__k{kbest}__lags{'-'.join(map(str, lag_set))}"
        f"__rolls{'-'.join(map(str, roll_set))}"
        f"__thr{base_threshold}"
        f"__nz{no_trade_cfg['lower']}_{no_trade_cfg['upper']}"
        f"__regime_{regime_mode}"
    )

    print(f"\nRunning {exp_name}")

    try:
        df_model = add_vixy_features_decision_safe(
            data.copy(),
            lag_set=lag_set,
            rolling_windows=roll_set,
            extra_market_lag=EXTRA_MARKET_LAG,
            target_forward_days=TARGET_FORWARD_DAYS
        )

        if USE_HMM:
            df_model = add_hmm_input_features(df_model)

        if USE_MARKOV_PROXY:
            df_model = add_markov_proxy_inputs(df_model)

        exclude_cols = {
            "Date", "vixy_close", "vixy_ret", "vixy_log_ret",
            "target_ret_fwd", "target_up"
        }

        vixy_engineered_cols = [
            c for c in df_model.columns
            if any(
                c.startswith(prefix)
                for prefix in [
                    "ret_lag_", "logret_lag_", "close_lag_",
                    "ret_mean_", "ret_std_", "price_ma_",
                    "price_vs_ma_", "cumret_"
                ]
            )
        ]

        hmm_helper_cols = [c for c in df_model.columns if c.startswith("hmm_")]
        markov_helper_cols = [c for c in df_model.columns if c.startswith("mk_")]

        macro_cols = [
            c for c in df_model.columns
            if c not in exclude_cols
            and c not in vixy_engineered_cols
            and c not in hmm_helper_cols
            and c not in markov_helper_cols
            and c not in ["hmm_state", "hmm_state_prob", "hmm_prev_state",
                          "markov_state", "markov_state_prob", "markov_prev_state"]
        ]

        if feature_mode == "macro_plus_ar":
            base_feature_cols = macro_cols + [c for c in vixy_engineered_cols if "lag_" in c]
        else:
            base_feature_cols = macro_cols + vixy_engineered_cols

        monthly_rows = []
        pred_rows = []
        processed_months = 0
        failed_months = 0

        for month_period in all_months:
            test_month_start = month_period.to_timestamp()
            test_month_end = (month_period + 1).to_timestamp()

            train_df = df_model[df_model["Date"] < test_month_start].copy()
            test_df = df_model[
                (df_model["Date"] >= test_month_start) &
                (df_model["Date"] < test_month_end)
            ].copy()

            if len(train_df) < MIN_TRAIN_ROWS or len(test_df) == 0:
                continue

            train_test_df = pd.concat([train_df, test_df], axis=0).sort_values("Date").reset_index(drop=True)

            try:
                if regime_mode == "hmm_filter":
                    train_test_df = fit_hmm_and_add_features(train_df=train_df, full_df=train_test_df)

                if regime_mode == "markov_filter":
                    train_test_df = fit_markov_proxy_and_add_features(train_df=train_df, full_df=train_test_df)

                train_aug = train_test_df[train_test_df["Date"] < test_month_start].copy()
                test_aug = train_test_df[
                    (train_test_df["Date"] >= test_month_start) &
                    (train_test_df["Date"] < test_month_end)
                ].copy()

                feature_cols = list(base_feature_cols)
                use_cols = ["Date", "target_up", "target_ret_fwd"] + feature_cols

                train_use = train_aug[use_cols].dropna(subset=["target_up"]).copy()
                test_use = test_aug[use_cols].dropna(subset=["target_up"]).copy()

                if len(train_use) == 0 or len(test_use) == 0:
                    failed_months += 1
                    continue

                X_train = train_use[feature_cols].copy()
                y_train = train_use["target_up"].astype(int).copy()
                X_test = test_use[feature_cols].copy()
                y_test = test_use["target_up"].astype(int).copy()

                pipe, k_used = build_pipeline(model, use_scaler, kbest, X_train.shape[1])
                pipe.fit(X_train, y_train)

                if hasattr(pipe, "predict_proba"):
                    prob_up = pipe.predict_proba(X_test)[:, 1]
                else:
                    z = pipe.decision_function(X_test)
                    prob_up = 1 / (1 + np.exp(-z))

                metrics = compute_metrics(y_test, prob_up, base_threshold)

                pred_df = test_use[["Date", "target_up", "target_ret_fwd"]].copy()
                pred_df["prob_up"] = prob_up

                if regime_mode == "hmm_filter":
                    regime_state_col = "hmm_state"
                    active_regime = 0
                    tmp_train = train_aug[[regime_state_col, "target_ret_fwd"]].dropna().copy()
                    if len(tmp_train) > 0:
                        ranked = (
                            tmp_train.groupby(regime_state_col)["target_ret_fwd"]
                            .apply(lambda x: np.mean(np.abs(x)))
                            .sort_values(ascending=False)
                        )
                        if len(ranked) > 0:
                            active_regime = ranked.index[0]

                    regime_info = test_aug[["Date", regime_state_col]].copy()
                    pred_df = pred_df.merge(regime_info, on="Date", how="left")
                    pred_df["signal"] = np.where(
                        (pred_df[regime_state_col] == active_regime) & (pred_df["prob_up"] >= no_trade_cfg["upper"]),
                        "LONG",
                        np.where(
                            (pred_df[regime_state_col] == active_regime) & (pred_df["prob_up"] <= no_trade_cfg["lower"]),
                            "SHORT",
                            "NO TRADE"
                        )
                    )

                elif regime_mode == "markov_filter":
                    regime_info = test_aug[["Date", "markov_state"]].copy()
                    pred_df = pred_df.merge(regime_info, on="Date", how="left")
                    pred_df["signal"] = np.where(
                        (pred_df["markov_state"] == 2) & (pred_df["prob_up"] >= no_trade_cfg["upper"]),
                        "LONG",
                        np.where(
                            (pred_df["markov_state"] == 2) & (pred_df["prob_up"] <= no_trade_cfg["lower"]),
                            "SHORT",
                            "NO TRADE"
                        )
                    )
                else:
                    pred_df["signal"] = np.where(
                        pred_df["prob_up"] >= no_trade_cfg["upper"], "LONG",
                        np.where(pred_df["prob_up"] <= no_trade_cfg["lower"], "SHORT", "NO TRADE")
                    )

                trade_stats = compute_trade_summary(pred_df)

                monthly_row = {
                    "experiment_name": exp_name,
                    "exp_id": exp_id,
                    "month": str(month_period),
                    "model_name": model_name,
                    "feature_mode": feature_mode,
                    "kbest": k_used,
                    "lags": ",".join(map(str, lag_set)),
                    "rolls": ",".join(map(str, roll_set)),
                    "base_threshold": base_threshold,
                    "no_trade_lower": no_trade_cfg["lower"],
                    "no_trade_upper": no_trade_cfg["upper"],
                    "regime_mode": regime_mode,
                    "train_rows": len(train_use),
                    "test_rows": len(test_use),
                    **metrics,
                    **trade_stats
                }
                monthly_rows.append(monthly_row)

                pred_df["experiment_name"] = exp_name
                pred_df["exp_id"] = exp_id
                pred_df["month"] = str(month_period)
                pred_df["model_name"] = model_name
                pred_df["feature_mode"] = feature_mode
                pred_rows.append(pred_df)

                processed_months += 1

            except Exception as e_month:
                failed_months += 1
                skip_log.append({
                    "level": "month",
                    "experiment_name": exp_name,
                    "month": str(month_period),
                    "reason": str(e_month)
                })
                continue

        if len(monthly_rows) == 0:
            skip_log.append({
                "level": "experiment",
                "experiment_name": exp_name,
                "month": "",
                "reason": "No valid monthly rows produced"
            })
            continue

        monthly_df_local = pd.DataFrame(monthly_rows)
        all_monthly_rows.extend(monthly_rows)

        if len(pred_rows) > 0:
            pred_df_local = pd.concat(pred_rows, axis=0, ignore_index=True)
            all_pred_rows.extend(pred_df_local.to_dict("records"))

        summary = {
            "experiment_name": exp_name,
            "exp_id": exp_id,
            "model_name": model_name,
            "feature_mode": feature_mode,
            "kbest": monthly_df_local["kbest"].iloc[0],
            "lags": ",".join(map(str, lag_set)),
            "rolls": ",".join(map(str, roll_set)),
            "base_threshold": base_threshold,
            "no_trade_lower": no_trade_cfg["lower"],
            "no_trade_upper": no_trade_cfg["upper"],
            "regime_mode": regime_mode,
            "months_processed": processed_months,
            "months_failed": failed_months,
            "accuracy_mean": monthly_df_local["accuracy"].mean(),
            "accuracy_std": monthly_df_local["accuracy"].std(),
            "balanced_accuracy_mean": monthly_df_local["balanced_accuracy"].mean(),
            "balanced_accuracy_std": monthly_df_local["balanced_accuracy"].std(),
            "f1_mean": monthly_df_local["f1"].mean(),
            "precision_mean": monthly_df_local["precision"].mean(),
            "recall_mean": monthly_df_local["recall"].mean(),
            "roc_auc_mean": monthly_df_local["roc_auc"].mean(),
            "trade_rate_mean": monthly_df_local["trade_rate"].mean(),
            "trade_hit_rate_mean": monthly_df_local["trade_hit_rate"].mean(),
            "monthly_fail_rate": failed_months / max(processed_months + failed_months, 1),
            "min_monthly_bacc": monthly_df_local["balanced_accuracy"].min(),
            "max_monthly_bacc": monthly_df_local["balanced_accuracy"].max(),
            "stability_score": np.nan
        }

        summary["stability_score"] = stability_score(pd.Series(summary))
        all_results_rows.append(summary)

        finished_exp += 1
        exp_elapsed = time.time() - exp_start_time
        total_elapsed = time.time() - run_start_time

        avg_per_exp = total_elapsed / finished_exp if finished_exp > 0 else np.nan
        remaining_exp = total_experiments - exp_id
        eta_seconds = avg_per_exp * remaining_exp if not np.isnan(avg_per_exp) else np.nan

        progress_rows.append({
            "exp_id": exp_id,
            "experiment_name": exp_name,
            "finished_experiments": finished_exp,
            "total_experiments": total_experiments,
            "progress_pct": round(100 * exp_id / total_experiments, 2),
            "exp_elapsed_sec": exp_elapsed,
            "avg_per_exp_sec": avg_per_exp,
            "elapsed_total_sec": total_elapsed,
            "eta_sec": eta_seconds,
            "elapsed_total_hms": seconds_to_hms(total_elapsed),
            "eta_hms": seconds_to_hms(eta_seconds),
            "stability_score": summary["stability_score"],
            "months_processed": processed_months
        })

        print(f"Finished {exp_name}")
        print(f"Progress: {exp_id}/{total_experiments} ({100 * exp_id / total_experiments:.2f}%)")
        print(f"This experiment time : {seconds_to_hms(exp_elapsed)}")
        print(f"Avg / experiment     : {seconds_to_hms(avg_per_exp)}")
        print(f"Elapsed total        : {seconds_to_hms(total_elapsed)}")
        print(f"Estimated remaining  : {seconds_to_hms(eta_seconds)}")
        print(f"Estimated total run  : {seconds_to_hms(total_elapsed + eta_seconds if not np.isnan(eta_seconds) else np.nan)}")
        print(f"Stability score      : {summary['stability_score']:.4f}")

        if finished_exp % SAVE_EVERY == 0:
            pd.DataFrame(all_results_rows).to_excel(
                os.path.join(OUTPUT_DIR, "intermediate_experiment_summary.xlsx"),
                index=False
            )
            pd.DataFrame(all_monthly_rows).to_excel(
                os.path.join(OUTPUT_DIR, "intermediate_monthly_detail.xlsx"),
                index=False
            )
            pd.DataFrame(all_pred_rows).to_excel(
                os.path.join(OUTPUT_DIR, "intermediate_predictions_detail.xlsx"),
                index=False
            )
            pd.DataFrame(skip_log).to_excel(
                os.path.join(OUTPUT_DIR, "intermediate_skip_log.xlsx"),
                index=False
            )
            pd.DataFrame(progress_rows).to_excel(
                os.path.join(OUTPUT_DIR, "run_progress_log.xlsx"),
                index=False
            )

    except Exception as e_exp:
        skip_log.append({
            "level": "experiment",
            "experiment_name": exp_name,
            "month": "",
            "reason": str(e_exp),
            "traceback": traceback.format_exc()
        })
        print(f"FAILED {exp_name}: {e_exp}")
        continue


# =========================================================
# STEP 6 — SAVE FINAL OUTPUTS
# =========================================================

results_df = pd.DataFrame(all_results_rows)
monthly_df = pd.DataFrame(all_monthly_rows)
preds_df = pd.DataFrame(all_pred_rows)
skip_df = pd.DataFrame(skip_log)
progress_df = pd.DataFrame(progress_rows)

results_file = os.path.join(OUTPUT_DIR, "full_experiment_summary.xlsx")
monthly_file = os.path.join(OUTPUT_DIR, "full_monthly_detail.xlsx")
preds_file = os.path.join(OUTPUT_DIR, "full_predictions_detail.xlsx")
skip_file = os.path.join(OUTPUT_DIR, "skip_log.xlsx")
progress_file = os.path.join(OUTPUT_DIR, "run_progress_log.xlsx")

results_df.to_excel(results_file, index=False)
monthly_df.to_excel(monthly_file, index=False)
preds_df.to_excel(preds_file, index=False)
skip_df.to_excel(skip_file, index=False)
progress_df.to_excel(progress_file, index=False)

print("\nSaved final files:")
print(results_file)
print(monthly_file)
print(preds_file)
print(skip_file)
print(progress_file)


# =========================================================
# STEP 7 — RANK EXPERIMENTS
# =========================================================

if len(results_df) > 0:
    ranked_df = results_df.sort_values(
        ["stability_score", "balanced_accuracy_mean", "roc_auc_mean", "trade_hit_rate_mean"],
        ascending=[False, False, False, False]
    ).reset_index(drop=True)

    ranked_file = os.path.join(OUTPUT_DIR, "ranked_experiments.xlsx")
    ranked_df.to_excel(ranked_file, index=False)

    print("\nTop 25 experiments:")
    print(ranked_df.head(25).to_string(index=False))
    print("\nSaved ranked experiments:", ranked_file)

    top5 = ranked_df.head(5)["experiment_name"].tolist()
    top5_monthly = monthly_df[monthly_df["experiment_name"].isin(top5)].copy()
    top5_monthly_file = os.path.join(OUTPUT_DIR, "top5_monthly_detail.xlsx")
    top5_monthly.to_excel(top5_monthly_file, index=False)

    print("Saved top-5 monthly detail:", top5_monthly_file)
else:
    print("\nNo successful experiments completed. Check skip_log.xlsx")